# **MODULE 03 - Additional-load calculation**
This Module calculates additional electricity loads associated with household activity profiles in a domestic context.
It computes EV charging, space heating, space cooling, domestic hot water (DHW), and cooking loads. 

It also returns dwelling-occupancy profiles accounting for each household member. It returns activity and related-energy consumption profiles and distribution of consumptions between household types.

## 3.1 Import libraries

In [ ]:
import os
import re
import random
import pandas as pd
import datetime
import numpy as np
from typing import Dict, Tuple
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


# Get the start time
start_time = datetime.datetime.now()
print(f"Start time : {start_time}")

## 3.2 Simulation parameters  
The following parameters control the additional-load calculations, including household size/type assumptions for heating.

In [ ]:
# -----------------------------------------------------------------------------
#   Sec 3.2 -- 1)   MAIN PARAMETERS
# -----------------------------------------------------------------------------
# Define the number of households to simulate
num_households = 146                                 # Align with ENG_02 unless a different local-energy-system size is required

# Region(s) to consider when building households and calculating consumption
selected_region = 'Cen'                             # Choose among (N-O, N-E, Cen, Sud, Isl)

household_profiles_name = f'profiles_{selected_region}_{num_households}_households'


# ------------- OCCUPANCY parameters ------------- 
# Probability that each activity is performed at home (p_home=1 at home, p_home=0 away; probabilities can be adjusted).
activity_map = {
    1:  {'cat': 'home',  'p_home': 1.00},           # Sleeping/Resting
    2:  {'cat': 'home',  'p_home': 1.00},           # Personal care & dressing
    3:  {'cat': 'maybe', 'p_home': 0.50},           # Eating        (DEPENDS ON WORK/STUDY)
    4:  {'cat': 'home',  'p_home': 1.00},           # Washing dishes
    5:  {'cat': 'home',  'p_home': 1.00},           # House cleaning
    6:  {'cat': 'home',  'p_home': 1.00},           # Cooking
    8:  {'cat': 'away',  'p_home': 0.00},           # Activities outside home
    9:  {'cat': 'home',  'p_home': 1.00},           # Doing laundry
    10: {'cat': 'home',  'p_home': 1.00},           # Watching TV/PC
    11: {'cat': 'away',  'p_home': 0.00},           # Sports
    12: {'cat': 'maybe', 'p_home': 0.10},           # Social interactions
    13: {'cat': 'maybe', 'p_home': 0.00},           # Health-related activities
    14: {'cat': 'maybe', 'p_home': 0.20},           # Work        (REMOTE OPTION)
    15: {'cat': 'home',  'p_home': 1.00},           # Reading
    16: {'cat': 'home',  'p_home': 1.00},           # Leisure at home
    17: {'cat': 'home',  'p_home': 0.99},           # Using PC
    18: {'cat': 'home',  'p_home': 0.99},           # Listening to music
    19: {'cat': 'home',  'p_home': 1.00},           # Home maintenance
    20: {'cat': 'home',  'p_home': 1.00},           # Ironing
    21: {'cat': 'maybe', 'p_home': 0.30},           # Studying      (REMOTE OPTION)
    22: {'cat': 'home',  'p_home': 0.50},           # Phone use     (DEPENDS ON WORK/STUDY)
    23: {'cat': 'maybe', 'p_home': 0.90},           # Other
}


# ------------- HEATING parameters ------------- 
# 1) Household-type specifications -> (family_size, min_apartment_area, max_apartment_area) (same dictionary used also in Module IV: check for consistency)
household_apartment_specs = {
    'single':                   (1,  30,  50),          # 1 person, 30–50 m²
    'couple_wo_children':       (2,  50,  70),          # 2 pp, 50–70 m²
    'couple_w_children_3':      (3,  70,  90),          # 3 pp, 70–90 m²
    'couple_w_children_4':      (4,  80, 120),          # 4 pp, 80–120 m²
    'couple_w_children_5':      (5, 100, 150),          # 5 pp, 100–150 m²
    'couple_w_children_6':      (6, 100, 150),          # 6 pp, 100–150 m²
    'single_parent_M_2':        (3,  70,  90),          # 2 pp, but assumed dwelling size as 3pp household type
    'single_parent_F_2':        (3,  70,  90),          # 2 pp, but assumed dwelling size as 3pp household type
}

# 2) Annual thermal-demand range per square metre (heating and cooling)
min_thermal_demand_per_m2 = 100                         # kWh_th/year/m² (min)
max_thermal_demand_per_m2 = 200                         # kWh_th/year/m² (max)

# 3) Consecutive absent timesteps required to turn the heat pump off (24 = 4 h)
min_absent_steps = 24



# -------------  COOKING parameters ------------- 
induction_ownership_probability = 1.00                  # (based on latest 2023 information, 6.7% of Italian households have induction hobs)
cold_season_induction_use_probability = 1.00            # if a household has an induction hob, it uses it every time it cooks outside summer
induction_consumption_kwh = 2.2 * 10 / 60               # average induction-hob consumption [kWh] = 2.2 [kW] * 10 [min] / 60 [min/h] = [kWh]
cooking_window_steps = 3                                # timestep window used to search for other cooking events close to an identified one
simultaneous_cooking_probability = 0.0                  # probability that multiple household members cook simultaneously (could be adjusted when considering flatmates situations, not included in the present study)




# ------------- EV CHARGING parameters ------------- 
ev_ownership_probability    = 1.00                      # EV ownership in the household
car_use_probability        = 1.00                       # probability of using the car for travel
km_per_step          = 6                                # average distance travelled in a 10-minute timestep
# initial_soc         = 0.65                            # % initial state of charge of EV battery
workplace_charging_probability = 0                      # probability of having a charging point at the workplace 

charging_efficiency           = 1                       # efficiency of the wallbox charging (can be adjusted)
min_dwell_steps      = 6                                # minimum continuous timesteps of EV plug-in at home to enable the home charging
home_charging_power_kw          = 3.0                   # kW home charger power
workplace_charging_power_kw        = 7.4                # kW workplace charger power

# battery capacity in kWh, specific consumption in kWh/km
ev_classes = {
    "segment_A":    {"descr":"city car",            "capacity": 20,  "cons_kwh_per_km": 0.15},         # type / battery capacity [kWh] / specific consumption [kWh/km]
    "segment_B":    {"descr":"compact cars",        "capacity": 50,  "cons_kwh_per_km": 0.16},
    "segment_C":    {"descr":"medium cars",         "capacity": 70,  "cons_kwh_per_km": 0.16}
    # "suv_compatti":  {"descr":"suv compatti",     "capacity": 60,  "cons_kwh_per_km": 0.16},         # compact SUV
    # "suv_medi":      {"descr":"suv medi",         "capacity": 80,  "cons_kwh_per_km": 0.18},         # medium size SUV
    # "segmento_D":    {"descr":"premium/lusso",    "capacity":110,  "cons_kwh_per_km": 0.18},         # premium-luxury EVs
}

# mapping household_type -> possible EV type ownership 
ev_classes_by_household_type = {
    "single":                       ["segment_A", "segment_B"],
    "couple_wo_children":           ["segment_B","segment_C"],
    "couple_w_children_3":          ["segment_B","segment_C"],
    "couple_w_children_4":          ["segment_C"],
    "couple_w_children_5":          ["segment_C"],
    "couple_w_children_6":          ["segment_C"]
}



# ------------- DHW parameters ------------- 
shower_ownership_probability = 1.0                  # es. 99% of households
# Shower probability conditional on a Personal-care event
morning_shower_use_probability  = 0.2               # probability of taking a shower when the "personal care" activity occurs in the morning (00:00-10:59)
daytime_shower_use_probability  = 0.1               # probability of taking a shower in the daytime (11:00-17:59)
evening_shower_use_probability  = 0.8               # probability of taking a shower in the evening (18:00-23:59)

simultaneous_shower_probability = 0.10              # simultaneity probability allowed (i.e., how probable is to have simultaneous showers between members within same household)
personal_care_cluster_max_gap_ts = 2                # Maximum distance, in timesteps, between two occurrences of "Personal care & dressing" for them to belong to the same personal-care event.




step_minutes = 10                                   # timestep duration [min]
max_shower_shift_steps = 72                         # maximum allowed postponement [ts] -> 72 ts = 12 h
alpha_util_boiler = 0.75                            # share of (Electric Storage Water Heater, ESWH) capacity that can be used within the household
gap_reset_cluster_ts = 12                           # after >=12 ts (=2h), duration to consider if include additional shower events within the same cluster
use_average_shower_for_shift = False                # if True, it uses average shower energy to calculate the minimum time required the ESWH to charge it back
                                                    # if False, it uses the shower energy consumed to calculate the time required to fill the ESWH back


# Dictionary DHW: household_type → (number of members, boiler capacity [kWh_th], charging power [kW])  
# >>> WARNING dictionary also defined in Module IV, need to be consistent <<<<<<<
dhw_specs = {
    'single':                   (1,  4.2,     1.5,    0.013),             # 1.35),     # ~80 L, 4.2 kWh, 1.5 kW, 1.3%/h static thermal losses,   1.35 kWh/24h (standing loss @T_hot=65°C and T_amb=20°C),
    'couple_wo_children':       (2,  5.2,     2,      0.013),             # 1.56),     # ~100 L, 5.2 kWh, 2 kW, 1.3%/h,                   1.56 kWh/24h,
    'couple_w_children_3':      (3,  10.5,    3,      0.007),             # 1.88),     # ~200 L, 10.5 kWh, 3 kW, 0.7%/h,                   1.88 kWh/24h
    'couple_w_children_4':      (4,  15.7,    4,      0.007),             # 2.6),      # ~300 L, 15-7 kWh, 4 kW, 0.7%/h,                 2.6 kWh/24h
    'couple_w_children_5':      (5,  15.7,    4,      0.007),             # 2.6),      # ~300 L, 15.7 kWh, 4 kW, 0.7%/h,                 2.6 kWh/24h
    'couple_w_children_6':      (6,  20.9,    6,      0.006),             # 3.6),      # 500 L, 20.9 kWh, 4 kW, 0.6%/h,                  3.6 kWh/24h,  --- (cfr noi 422 L in household da 4, caso ampio) stessa taglia delle households molto grandi
    'single_parent_M_2':        (3,  5.2,     2,      0.013),             # 1.56),     # 2-member single-parent household -> use the size assumed for 3-person households
    'single_parent_F_2':        (3,  5.2,     2,      0.013),             # 1.56),
}


# --- DHW demand calculation for one shower ---
"""
    The following assumptions are used to calculate
    the thermal energy associated with a single shower,
    in order to define the minimum and maximum values below.

    Assumptions:
        - average shower duration:                              6-8 min
        - shower flow rate:                                     8-10 L/min
        - water consumption per shower:                         60 L/shower
          (efficient case; otherwise up to 100 L/shower)

        - specific heat capacity of water:                      4.186 kJ/kg/K
        - final water temperature supplied to the user:         45°C
        - inlet/cold water temperature:                         10°C
          (in summer, 15°C could be assumed)
        - water temperature difference from boiler:             35°C
          (T_final - T_inlet)
        - basin water consumption:                              total daily amount
      
    Q_th = V_H2O * c_H2O * DeltaT
         = 60 L/shower * 4.184 kJ/kg/K * 35 K
         = 8790 kJ / 3600
         = 2.44 kWh/shower

    Average values:
        - Efficient shower
          (10 min × 6-6.3 L/min = 60-63 L):
          ~1.7-2.2 kWh_th
        - Standard shower
          (8-10 min × 8-10 L/min = 64-100 L):
          ~1.9-3.5 kWh_th

    Empirical values:
        - 1.4-1.9 kWh per shower, from Tiefenbach et al. (2017), Swiss Federal Office of Energy / ETH Zurich,based on readings from 700 homes in Zurich          --> https://pubdb.bfe.admin.ch/en/publication/download/8561
        - 0.9-2.5 kWh per shower, from Tiefenbach et al. (2019), ETH Zurich, based on a sample of 19,600 shower readings in hotels --> https://www.alexandria.unisg.ch/server/api/core/bitstreams/32e56360-b2c6-4c67-b81e-f08038871dff/content

          The hotel range may be wider because users are generally less attentive
          to consumption when they do not pay directly for it.
"""

min_shower_consumption_kwh_th = 1.4      # kWh for shower (from work Swiss Fed Office Energy, ETH over 700 dwellings, 2017) --> https://pubdb.bfe.admin.ch/en/publication/download/8561
max_shower_consumption_kwh_th = 1.9      # kWh for shower (from work Swiss Fed Office Energy, ETH over 700 dwellings, 2017) --> https://pubdb.bfe.admin.ch/en/publication/download/8561
dhw_boiler_efficiency = 1.00              # originally = 0.98, but assumed 1, so kept thermal demand and then conversion in Module IV # kWh_th/kWh_el  (but WARNING thermal losses not considered)
reference_shower_consumption_kwh_th = (min_shower_consumption_kwh_th + max_shower_consumption_kwh_th) / 2  # average consumption shower for conservative approach



# -----------------------------------------------------------------------------
#   Sec 3.2 -- 2)  Seasonal periods to calculate
# -----------------------------------------------------------------------------

selectable_periods = (
    "winter",
    "spring",
    "summer",
    "autumn",
)


# -----------------------------------------------------------------------------
#   Sec 3.2 -- 3)   Plot-selection parameters
# -----------------------------------------------------------------------------

# Seasonal profiles used for the combined daily activity-consumption plot.
# Use "all", one season such as "winter", or a sequence such as ("winter", "summer").
DAILY_ACTIVITY_CONSUMPTION_PLOT_SEASONS = "all"

# Day position inside each seven-day seasonal profile: 0 = first day, ..., 6 = seventh day.
DAILY_ACTIVITY_CONSUMPTION_DAY_INDEX = 0

# Optional explicit date by season; an explicit date overrides the day index.
DAILY_ACTIVITY_CONSUMPTION_DATE_BY_SEASON = {}

# Layouts generated for weekly stacked-consumption plots.
STACKED_CONSUMPTION_PLOT_LAYOUTS = (
    "two_column",
)

# Sheet type read from the conventional-load workbook.
BASE_LOAD_SHEET_KIND = "TOT"

# Name used for the conventional component in combined plots.
BASE_LOAD_LABEL = "Conventional"

# Percentiles and percentile bands shown in annual-distribution plots.
PERCENTILE_LEVELS = (
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
)

PERCENTILE_BANDS = (
    (10, 90),
    (20, 80),
    (30, 70),
    (40, 60),
)

## 3.3 Directories

In [ ]:
# -----------------------------------------------------------------------------
#   General folders
# -----------------------------------------------------------------------------

input_folder = "input"
output_folder = "output"


# -----------------------------------------------------------------------------
#   Input paths
# -----------------------------------------------------------------------------

household_profiles_input_dir = os.path.join(output_folder,    "02_household_activity_profiles_Xls",)
household_profiles_file = os.path.join(household_profiles_input_dir,    f"{household_profiles_name}.xlsx",)

additional_consumption_input_folder = os.path.join(input_folder,    "03_additional_cons",)

cooling_input_folder = os.path.join(additional_consumption_input_folder,    "Cooling",)
cooling_profiles_file = os.path.join(cooling_input_folder,    "cooling_profiles_4_seasons.xlsx",)


heating_coefficients_path = os.path.join(additional_consumption_input_folder,    "Heating",    "Coefficients_heating.xlsx",)
temperature_data_path = os.path.join(additional_consumption_input_folder,    "Heating",    "climate_data_58091_2023.xlsx",)
thermal_profile_cop_excel_path = os.path.join(additional_consumption_input_folder,    "Heating",    "normalized_thermal_profile&COP.xlsx",)

base_consumption_input_xls_folder = os.path.join(output_folder,    "02_base_consumption_Xls",)

aggregated_base_load_file = os.path.join(base_consumption_input_xls_folder, f"{num_households}_households_{selected_region}_ts_kWh_aggregated_households_base.xlsx" )
base_load_total_only_file = os.path.join(base_consumption_input_xls_folder, f"{num_households}_households_{selected_region}_ts_kWh_base_total_only.xlsx")

BASE_LOAD_FULL_EXCEL_PATH = aggregated_base_load_file
BASE_LOAD_TOTAL_ONLY_EXCEL_PATH = base_load_total_only_file


# -----------------------------------------------------------------------------
#   Output paths
# -----------------------------------------------------------------------------

additional_consumption_output_xls_folder = os.path.join(
    output_folder,
    "03_additional_consumption_Xls",
)

additional_consumption_output_pictures_folder = os.path.join(
    output_folder,
    "03_additional_consumption_Pics",
    f"{num_households}_households_{selected_region}",
)

additional_weekly_aggregated_file = os.path.join(
    additional_consumption_output_xls_folder,
    (
        f"{num_households}_households_{selected_region}_"
        f"weekly_aggregated_ts_kWh_additional_loads.xlsx"
    ),
)

additional_annual_values_file = os.path.join(
    additional_consumption_output_xls_folder,
    (
        f"{num_households}_households_{selected_region}_"
        f"annual_values_kWh_additional_loads.xlsx"
    ),
)

additional_annual_statistics_file = os.path.join(
    additional_consumption_output_xls_folder,
    (
        f"{num_households}_households_{selected_region}_"
        f"annual_statistics_additional_loads.xlsx"
    ),
)


# -----------------------------------------------------------------------------
#   Create output folders only
# -----------------------------------------------------------------------------

for folder in (
    output_folder,
    additional_consumption_output_xls_folder,
    additional_consumption_output_pictures_folder,
):
    os.makedirs(folder, exist_ok=True)

## 3.4 Selectable sections

In [ ]:
# =============================================================================
#   Sec 3.4 -- SELECTABLE EXECUTION FLAGS
# =============================================================================

# -----------------------------------------------------------------------------
#   1) Main execution stages
# -----------------------------------------------------------------------------
RUN_ADDITIONAL_LOADS                        = True                      # Sec 3.7: Calculate and save occupancy, DHW, cooking, EV, cooling and heating profiles.
RUN_ADDITIONAL_LOAD_PLOTS                   = True                     # Sec 3.8: Read existing MODULE_03 outputs and generate the selected plots

RUN_REBUILD_THERMAL_PROFILE                 = True                     # Sec 3.7: Rebuild and save the normalized annual space heating-demand profile from climate data.

# -----------------------------------------------------------------------------
#   2) Plot-selection flags
#   Used only when RUN_ADDITIONAL_LOAD_PLOTS = True
# -----------------------------------------------------------------------------
PLOT_WEEKLY_ADDITIONAL_STACKED              = True                      # Sec 3.8: Plot the stacked weekly profile containing additional loads only.
PLOT_STACKED_CONV_PLUS_ADDITIONAL           = True                      # Sec 3.8: Plot the stacked weekly profile containing conventional and additional loads.

RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT         = True                      # Sec 3.8: Plot one daily activity profile above the corresponding total-consumption profile.

PLOT_ANNUAL_PERCENTILES_ALL_HOUSEHOLDS      = False                     # Sec 3.8: Plot one annual percentile distribution for each load across all households.
PLOT_ANNUAL_PERCENTILES_BY_HOUSEHOLD_TYPE   = True                      # Sec 3.8: Plot annual percentile distributions separated by household type.


# -----------------------------------------------------------------------------
#   3) Plot-output flags
# -----------------------------------------------------------------------------
SHOW_STACKED_CONSUMPTION_TITLE              = False                     # Sec 3.8: Show titles in weekly stacked-consumption figures.
SHOW_DAILY_ACTIVITY_CONSUMPTION_TITLE       = False                     # Sec 3.8: Show the title in the combined daily activity-consumption figure.

SAVE_STACKED_CONSUMPTION_PDF                = False                      # Sec 3.8: Save weekly stacked-consumption plots as both PNG and PDF.
SAVE_DAILY_ACTIVITY_CONSUMPTION_PDF         = False                      # Sec 3.8: Save combined daily activity-consumption plots as both PNG and PDF.
SAVE_ANNUAL_PERCENTILE_PDF                  = False                     # Sec 3.8: Save annual percentile plots as both PNG and PDF


# -----------------------------------------------------------------------------
#   4) Optional statistical outputs
# -----------------------------------------------------------------------------
SAVE_ANNUAL_STATISTICS_TABLE                = True                      # Save descriptive annual statistics overall and by household type.


# -----------------------------------------------------------------------------
#   5) Optional diagnostic check
# -----------------------------------------------------------------------------
# Read-only audit of already generated EV mobility profiles.
# Does NOT rebuild or overwrite MODULE_03 demand profiles.
RUN_EV_TRIP_EXTREME_AUDIT                   = True

## 3.5 Dictionaries

In [ ]:
season_english_dict = {
    'winter': 'Winter',
    'spring': 'Spring',
    'summer': 'Summer',
    'autumn': 'Autumn'
}

 

# Mapping season -> (start_date, end_date). WARNING: check consistency with Module I selected periods.
season_periods = {
    'winter': ('2023-02-15 00:00:00', '2023-02-21 23:50:00'),
    'spring': ('2023-04-19 00:00:00', '2023-04-25 23:50:00'),
    'summer': ('2023-07-12 00:00:00', '2023-07-18 23:50:00'),
    'autumn': ('2023-10-11 00:00:00', '2023-10-17 23:50:00'),
}


# Full output column -> abbreviation used in files/titles
categories_map = {
    'DHW_kWh_el_annual':                    'DHW',
    'Cooking_kWh_el_annual':                'Cooking',
    'EV_home_charging_kWh_el_annual':       'EV',
    'annual_km':                            'km',
    'Heating_kWh_th_annual':                'Heating Th',
    'Cooling_kWh_el_annual':                'Cooling'
}



# Compatibility dictionaries and helpers for ITA/ENG labels.
# The module uses English canonical names internally, but these helpers allow it
# to read legacy Italian outputs when necessary.
HOUSEHOLD_TYPE_ALIASES = {
    'single': 'single', 'membro_isolato': 'single',
    'couple_wo_child': 'couple_wo_children', 'couple_wo_children': 'couple_wo_children', 'coppia_SF': 'couple_wo_children',
    'couple_w_child_3': 'couple_w_children_3', 'couple_w_children_3': 'couple_w_children_3', 'coppia_CF_3': 'couple_w_children_3',
    'couple_w_child_4': 'couple_w_children_4', 'couple_w_children_4': 'couple_w_children_4', 'coppia_CF_4': 'couple_w_children_4',
    'couple_w_child_5': 'couple_w_children_5', 'couple_w_children_5': 'couple_w_children_5', 'coppia_CF_5': 'couple_w_children_5',
    'couple_w_child_6': 'couple_w_children_6', 'couple_w_children_6': 'couple_w_children_6', 'coppia_CF_6': 'couple_w_children_6',
    'mono_M_2': 'single_parent_M_2', 'single_parent_M_2': 'single_parent_M_2', 'mono_CF_M_2': 'single_parent_M_2',
    'mono_F_2': 'single_parent_F_2', 'single_parent_F_2': 'single_parent_F_2', 'mono_CF_F_2': 'single_parent_F_2',
}

SEASON_ALIASES = {
    'winter': 'winter', 'inverno': 'winter', 'inv': 'winter',
    'spring': 'spring', 'primavera': 'spring', 'pri': 'spring',
    'summer': 'summer', 'estate': 'summer', 'est': 'summer',
    'autumn': 'autumn', 'fall': 'autumn', 'autunno': 'autumn', 'aut': 'autumn',
}

COOLING_SEASON_COLUMN_ALIASES = {
    'winter': ['winter', 'inverno', 'Winter', 'Inverno'],
    'spring': ['spring', 'primavera', 'Spring', 'Primavera'],
    'summer': ['summer', 'estate', 'Summer', 'Estate'],
    'autumn': ['autumn', 'fall', 'autunno', 'Autumn', 'Fall', 'Autunno'],
}



# ------------------ Activity dictionaries for combined daily activity + consumption plots
# These labels are intentionally kept in Italian as internal keys because they
# match the ISTAT-based activity classification used in ENG_01. The legend uses
# the English labels through activity_english_dict.

aggregated_activities = [
    'Dormire/Riposo',
    'Igiene e vestirsi',
    'Mangiare',
    'Lavare stoviglie',
    'Pulizia casa',
    'Cucinare',
    'Spostamenti',
    'Attività fuori casa',
    'Lavare bucato',
    'Guardare TV/PC',
    'Sport',
    'Contatti sociali',
    'Salute',
    'Lavoro',
    'Lettura',
    'Tempo libero a casa',
    'Uso PC',
    'Ascoltare musica',
    'Manutenzione domestica',
    'Stiratura',
    'Studio',
    'Telefono',
    'Altro'
]

activity_english_dict = {
    'Dormire/Riposo': 'Sleeping/Resting',
    'Igiene e vestirsi': 'Personal care & dressing',
    'Mangiare': 'Eating',
    'Lavare stoviglie': 'Washing dishes',
    'Pulizia casa': 'House cleaning',
    'Cucinare': 'Cooking',
    'Spostamenti': 'Travelling/Commuting',
    'Attività fuori casa': 'Activities outside home',
    'Lavare bucato': 'Doing laundry',
    'Guardare TV/PC': 'Watching TV/PC',
    'Sport': 'Sports',
    'Contatti sociali': 'Social interactions',
    'Salute': 'Health-related activities',
    'Lavoro': 'Work',
    'Lettura': 'Reading',
    'Tempo libero a casa': 'Leisure at home',
    'Uso PC': 'Using PC',
    'Ascoltare musica': 'Listening to music',
    'Manutenzione domestica': 'Home maintenance',
    'Stiratura': 'Ironing',
    'Studio': 'Studying',
    'Telefono': 'Phone use',
    'Altro': 'Other'
}

activity_color_map = {
    # Rest / personal routines
    'Dormire/Riposo':            '#1b2a6b',
    'Igiene e vestirsi':         '#4c72b0',
    'Mangiare':                  '#86b6f6',

    # Domestic work
    'Lavare stoviglie':          '#56b4e9',
    'Pulizia casa':              '#b3d7e8',
    'Cucinare':                  '#009e73',
    'Lavare bucato':             '#e69f00',
    'Stiratura':                 '#44aa99',
    'Manutenzione domestica':    '#117733',

    # Mobility / outside home
    'Spostamenti':               '#7fbc41',
    'Attività fuori casa':       '#f0e442',

    # Leisure / media
    'Guardare TV/PC':            '#d55e00',
    'Tempo libero a casa':       '#dd3497',
    'Uso PC':                    '#f781bf',
    'Ascoltare musica':          '#fbb4ae',
    'Lettura':                   '#b07cc6',
    'Telefono':                  '#2a9d8f',

    # Social / health / sport
    'Sport':                     '#cc6677',
    'Contatti sociali':          '#882255',
    'Salute':                    '#aa4499',

    # Work / study
    'Lavoro':                    '#6a3d9a',
    'Studio':                    '#88ccee',

    # Residual class
    'Altro':                     '#999999'
}

# Activity-code mapping used by household activity-profile workbooks.
# The Excel file stores numerical activity codes, while the plot uses
# the ISTAT activity labels above as internal categories.
activity_code_to_label = {
    1:  'Dormire/Riposo',
    2:  'Igiene e vestirsi',
    3:  'Mangiare',
    4:  'Lavare stoviglie',
    5:  'Pulizia casa',
    6:  'Cucinare',
    7:  'Spostamenti',
    8:  'Attività fuori casa',
    9:  'Lavare bucato',
    10: 'Guardare TV/PC',
    11: 'Sport',
    12: 'Contatti sociali',
    13: 'Salute',
    14: 'Lavoro',
    15: 'Lettura',
    16: 'Tempo libero a casa',
    17: 'Uso PC',
    18: 'Ascoltare musica',
    19: 'Manutenzione domestica',
    20: 'Stiratura',
    21: 'Studio',
    22: 'Telefono',
    23: 'Altro',
}


# Fixed colours between seasons and scenarios (to be equal to colours in Module IV)
STACKED_CONSUMPTION_COLOR_MAP = {
    "Conventional":      "#B3B3B3",
    "DHW":               "#0072B2",
    "Cooking":           "#E69F00",
    "Cooling":           "#009E73",
    "EV":                "#D55E00",
    "Heating":           "#7E57C2",
}

STACKED_CONSUMPTION_ORDER = [
    "Conventional",
    "DHW",
    "Cooking",
    "Cooling",
    "EV",
    "Heating",
]

# Annual metrics used by percentile plots and statistical tables.
ANNUAL_METRIC_SPECS = {
    "DHW_kWh_el_annual": {
        "label": "DHW",
        "file_tag": "DHW",
        "unit": "kWh_el/year",
    },
    "Cooking_kWh_el_annual": {
        "label": "Cooking",
        "file_tag": "Cooking",
        "unit": "kWh_el/year",
    },
    "EV_home_charging_kWh_el_annual": {
        "label": "EV home charging",
        "file_tag": "EV_home_charging",
        "unit": "kWh_el/year",
    },
    "annual_km": {
        "label": "Distance travelled",
        "file_tag": "annual_km",
        "unit": "km/year",
    },
    "Heating_kWh_th_annual": {
        "label": "Heating demand",
        "file_tag": "Heating_thermal",
        "unit": "kWh_th/year",
    },
    "Cooling_kWh_el_annual": {
        "label": "Cooling",
        "file_tag": "Cooling",
        "unit": "kWh_el/year",
    },
}


# Reference household counts associated with the tested grid archetypes.
GRID_ARCHETYPE_HOUSEHOLD_OPTIONS = {
    "rural": (8, 13, 14),
    "mixed": (57,),
    "urban": (144, 146),
}
# To print the dictionary:
selected_grid_archetype = next(
    (
        archetype
        for archetype, household_counts
        in GRID_ARCHETYPE_HOUSEHOLD_OPTIONS.items()
        if num_households in household_counts
    ),
    "custom",
)

print(
    f"Selected local system: {num_households} households "
    f"({selected_grid_archetype} grid archetype)"
)



STACKED_CONSUMPTION_LAYOUT_PRESETS = {
    "single_column": {
        "fig_size": (3.55, 2.80),
        "font_size": 8.0,
        "label_font_size": 8.5,
        "tick_font_size": 7.2,
        "legend_font_size": 6.8,
        "title_font_size": 9.0,
        "legend_position": "bottom",
        "legend_ncol": 3,
    },
    "two_column": {
        "fig_size": (7.20, 3.60),
        "font_size": 9.5,
        "label_font_size": 10.5,
        "tick_font_size": 9.0,
        "legend_font_size": 8.0,
        "title_font_size": 11.0,
        "legend_position": "right",
        "legend_ncol": 1,
    },
}

## 3.6 Functions

### 3.6.1 F(x) for ITA/ENG dictionaries

In [ ]:
#-------------------------------------------------------------------
#       Sec. 3.6 -- 1) FUNCTIONS FOR CONVERTING DICTIONARIES ITA-ENG
#-------------------------------------------------------------------

# Function to return English household type label
def canonical_household_type(household_type):
    """Return the canonical English household-type label used by ENG_02."""
    household_type = str(household_type)
    return HOUSEHOLD_TYPE_ALIASES.get(household_type, household_type)


# Function to return ENG season key
def canonical_season(season):
    """Return the canonical English season key."""
    return SEASON_ALIASES.get(str(season).casefold(), str(season))


# Function to normalize columns in household profiles
def normalize_household_profile_column_name(column_name, default_season=None):
    """Normalize household-profile columns to Household_X_Member_Y_<season>_<type>."""
    if not isinstance(column_name, str) or column_name == 'Datetime':
        return column_name

    match = re.match(r'^(Household|Nucleo)_(\d+)_(Member|Membro)_(\d+)_(.+)$', column_name)
    if not match:
        return column_name

    household_number = match.group(2)
    member_number = match.group(4)
    tail = match.group(5)

    season_candidates = sorted(SEASON_ALIASES.keys(), key=len, reverse=True)
    for season_candidate in season_candidates:
        if tail == season_candidate or tail.startswith(season_candidate + '_'):
            season_key = canonical_season(season_candidate)
            raw_household_type = tail[len(season_candidate):].lstrip('_')
            household_type = canonical_household_type(raw_household_type)
            return f'Household_{household_number}_Member_{member_number}_{season_key}_{household_type}'

    if default_season is not None:
        season_key = canonical_season(default_season)
        household_type = canonical_household_type(tail)
        return f'Household_{household_number}_Member_{member_number}_{season_key}_{household_type}'

    return column_name


# Function to normalize household profiles in dataset
def normalize_household_profile_columns(df, default_season=None):
    """Normalize all household-profile columns in a DataFrame."""
    df = df.copy()
    df.columns = [normalize_household_profile_column_name(c, default_season) for c in df.columns]
    return df


# Function to find column using english keys
def find_compatible_column(df, canonical_key, aliases_dict):
    """Find a dataframe column using English canonical keys plus legacy aliases."""
    aliases = aliases_dict.get(canonical_key, [canonical_key])
    normalized = {re.sub(r'[^a-z0-9]+', '', str(c).casefold()): c for c in df.columns}
    for alias in aliases:
        key = re.sub(r'[^a-z0-9]+', '', str(alias).casefold())
        if key in normalized:
            return normalized[key]
    raise KeyError(f"No compatible column found for '{canonical_key}'. Available columns: {list(df.columns)}")

### 3.6.2 F(x) defining base loads for the overall plot

In [ ]:
#-------------------------------------------------------------------
#       Sec. 3.6 -- 2) FUNCTIONS FOR DEFINING BASE LOAD
#-------------------------------------------------------------------
# Function to normalize Excel sheet names
def _normalize_excel_sheet_name(name):
    """
    Normalize Excel sheet names:
      - ignore case
      - collapse multiple spaces
      - strip leading/trailing spaces
    """
    s = str(name).strip()
    s = re.sub(r"\s+", " ", s)
    return s.casefold()


# Function to find the conventional-load workbook sheets robustly
def _find_base_load_sheet_name(base_load_xls, season, sheet_kind="TOT"):
    """
    Find the base-load sheet robustly with respect to:
      - Italian/English labels
      - case
      - repeated spaces

    Accepted examples:
      inverno TOT
      Winter TOT
      winter TOT
      WINTER TOT
    """
    season = str(season).strip().lower()
    sheet_kind = str(sheet_kind).strip()

    season_aliases = {
        "winter": ["winter", "inverno"],
        "spring": ["spring", "primavera"],
        "summer": ["summer", "estate"],
        "autumn": ["autumn", "fall", "autunno"],
    }

    aliases = season_aliases.get(season, [season])

    candidate_names = []
    for alias in aliases:
        candidate_names.append(f"{alias} {sheet_kind}")
        candidate_names.append(f"{alias.capitalize()} {sheet_kind}")
        candidate_names.append(f"{alias.upper()} {sheet_kind}")

    available_norm = {
        _normalize_excel_sheet_name(sheet): sheet
        for sheet in base_load_xls.sheet_names
    }

    for candidate in candidate_names:
        candidate_norm = _normalize_excel_sheet_name(candidate)
        if candidate_norm in available_norm:
            return available_norm[candidate_norm]

    print(
        f"[WARN][BASE LOAD] No sheet found for season='{season}', "
        f"sheet_kind='{sheet_kind}'."
    )
    print(f"[WARN][BASE LOAD] Tried names: {candidate_names}")
    print(f"[WARN][BASE LOAD] Available sheets: {base_load_xls.sheet_names}")

    return None


# Function to keep only numeric columns
def _extract_numeric_base_load_df(df_base):
    """
    Keep only numeric columns and discard textual columns/metadata.
    """
    df_base_num = df_base.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    if df_base_num.shape[1] == 0:
        return None

    return df_base_num


# Function to automatically create the aggregated file if missing
def _build_base_load_total_only_excel_if_missing(
    full_xlsx,
    total_only_xlsx,
    sheet_kind="TOT",
    value_col=None,
):
    """
    Automatically create the lightweight Excel file of aggregated base loads
    if it does not already exist.

    Logic:
      1) if the aggregated file exists, use it;
      2) if it does not exist, try to create it from the full file;
      3) if the full file is also missing, return None.
    """
    if value_col is None:
        value_col = BASE_LOAD_LABEL

    # If the lightweight file already exists, do not recreate it.
    if os.path.exists(total_only_xlsx):
        print(f"[INFO][BASE LOAD] Using existing aggregated file: {total_only_xlsx}")
        return total_only_xlsx

    # If the full file is missing, the aggregated one cannot be created.
    if not os.path.exists(full_xlsx):
        print(f"[WARN][BASE LOAD] Aggregated file not found: {total_only_xlsx}")
        print(f"[WARN][BASE LOAD] Full file not found: {full_xlsx}")
        print("[WARN][BASE LOAD] The stacked conventional+additional plot will be skipped.")
        return None

    print("[INFO][BASE LOAD] Aggregated file not found.")
    print(f"[INFO][BASE LOAD] Automatically creating: {total_only_xlsx}")
    print(f"[INFO][BASE LOAD] Full source file: {full_xlsx}")

    xls = pd.ExcelFile(full_xlsx)
    os.makedirs(os.path.dirname(total_only_xlsx), exist_ok=True)

    created_any_sheet = False

    with pd.ExcelWriter(total_only_xlsx, engine="openpyxl") as writer:
        for season in selectable_periods:
            sheet_name = _find_base_load_sheet_name(
                base_load_xls=xls,
                season=season,
                sheet_kind=sheet_kind,
            )

            if sheet_name is None:
                print(f"[WARN][BASE LOAD] Skipping '{season}': base-load sheet not found.")
                continue

            df_base = pd.read_excel(
                xls,
                sheet_name=sheet_name,
                index_col=0,
            )

            if df_base.empty:
                print(f"[WARN][BASE LOAD] Skipping '{season}': sheet '{sheet_name}' is empty.")
                continue

            # Try to preserve the DatetimeIndex.
            idx_dt = pd.to_datetime(df_base.index, errors="coerce")
            if idx_dt.notna().sum() == len(idx_dt):
                df_base.index = idx_dt

            df_base_num = _extract_numeric_base_load_df(df_base)

            if df_base_num is None:
                print(
                    f"[WARN][BASE LOAD] Skipping '{season}': no numeric columns "
                    f"found in sheet '{sheet_name}'."
                )
                continue

            # This calculation can be computationally expensive, but it is only
            # performed when the aggregated file does not already exist.
            base_total = df_base_num.sum(axis=1).rename(value_col)

            df_out = pd.DataFrame({value_col: base_total})

            # Save the profile sheets using the same naming convention as the notebook.
            out_sheet = f"{season} {sheet_kind}"
            df_out.to_excel(writer, sheet_name=out_sheet)
            created_any_sheet = True

            print(
                f"[OK][BASE LOAD] Aggregated '{sheet_name}' -> '{out_sheet}' "
                f"({len(df_out)} time steps)"
            )

    if not created_any_sheet:
        print("[WARN][BASE LOAD] No aggregated sheet was created.")
        if os.path.exists(total_only_xlsx):
            os.remove(total_only_xlsx)
        return None

    print(f"[OK][BASE LOAD] Saved aggregated file: {total_only_xlsx}")
    return total_only_xlsx


# Function returning the aggregated conventional-load file
def _resolve_base_load_excel_path():
    """
    Return the Excel file to use for the base+additional plot.

    Automatic logic:
      1) look for the already aggregated lightweight file;
      2) if it does not exist, create it from the full file;
      3) if the full file does not exist either, skip the plot.
    """
    # if not PLOT_STACKED_CONV_PLUS_ADDITIONAL:
    #     return None

    return _build_base_load_total_only_excel_if_missing(
        full_xlsx=BASE_LOAD_FULL_EXCEL_PATH,
        total_only_xlsx=BASE_LOAD_TOTAL_ONLY_EXCEL_PATH,
        sheet_kind=BASE_LOAD_SHEET_KIND,
        value_col=BASE_LOAD_LABEL,
    )


# Function to read the conventional-load file
def _read_base_load_total_for_season(
    base_load_xls,
    season,
    target_index,
    sheet_kind="TOT",
):
    """
    Read the aggregated profile for one season from the lightweight Excel file.

    Accept sheet names both in Italian and English:
      - inverno TOT / Winter TOT / winter TOT
      - primavera TOT / Spring TOT / spring TOT
      - estate TOT / Summer TOT / summer TOT
      - autunno TOT / Autumn TOT / autumn TOT

    The expected file is the already aggregated one, with a single column:
      Conventional
    """
    sheet_name = _find_base_load_sheet_name(
        base_load_xls=base_load_xls,
        season=season,
        sheet_kind=sheet_kind,
    )

    if sheet_name is None:
        return None

    df_base = pd.read_excel(
        base_load_xls,
        sheet_name=sheet_name,
        index_col=0,
    )

    if df_base.empty:
        print(f"[WARN][BASE LOAD] Sheet '{sheet_name}' is empty.")
        return None

    idx_dt = pd.to_datetime(df_base.index, errors="coerce")
    if idx_dt.notna().sum() == len(idx_dt):
        df_base.index = idx_dt

    df_base_num = _extract_numeric_base_load_df(df_base)

    if df_base_num is None:
        print(f"[WARN][BASE LOAD] No numeric columns found in sheet '{sheet_name}'.")
        return None

    # Ideal case: lightweight file already aggregated with a Conventional column.
    if BASE_LOAD_LABEL in df_base_num.columns:
        base_total = df_base_num[BASE_LOAD_LABEL].copy()
        base_total.name = BASE_LOAD_LABEL

    # Robust fallback: if for some reason a full file is read, sum all numeric columns.
    else:
        print(
            f"[WARN][BASE LOAD] Column '{BASE_LOAD_LABEL}' not found in sheet "
            f"'{sheet_name}'. Using fallback: sum of all numeric columns."
        )
        base_total = df_base_num.sum(axis=1)
        base_total.name = BASE_LOAD_LABEL

    # Align the base-load index with the additional-load index.
    if isinstance(target_index, pd.DatetimeIndex) and isinstance(base_total.index, pd.DatetimeIndex):
        base_total_reindexed = base_total.reindex(target_index)

        if base_total_reindexed.isna().mean() > 0.5 and len(base_total) == len(target_index):
            base_total = pd.Series(
                base_total.to_numpy(),
                index=target_index,
                name=BASE_LOAD_LABEL,
            )
        else:
            base_total = base_total_reindexed

    elif len(base_total) == len(target_index):
        base_total = pd.Series(
            base_total.to_numpy(),
            index=target_index,
            name=BASE_LOAD_LABEL,
        )

    else:
        print(
            f"[WARN][BASE LOAD] Cannot align the base load for '{season}'. "
            f"Base length={len(base_total)}, additional length={len(target_index)}."
        )
        return None

    base_total = pd.to_numeric(base_total, errors="coerce").fillna(0.0)

    return base_total


# Function to format dates printed on consumption plots
def _format_day_month_en(x, pos=None):
    """
    Format x-axis dates as DD-Mmm without year, e.g. 15-Feb.
    Does not depend on the system locale.
    """
    month_abbr = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ]

    dt = mdates.num2date(x)
    return f"{dt.day:02d}-{month_abbr[dt.month - 1]}"


# Function to print additional+conventional consumption plots
def plot_stacked_consumption_for_paper(
        df_plot,
        title,
        file_name,
        output_folder,
        column_order=None,
        color_map=None,
        show_title=False,
        fig_size=(7.20, 3.70),
        font_size=9.5,
        label_font_size=10.5,
        tick_font_size=9,
        legend_font_size=8,
        title_font_size=11,
        legend_position="right",
        legend_ncol=1,
        y_label="Energy demand [kWh]",
        alpha=0.95,
        save_pdf=True):
    """
    Journal-ready stacked-area plot for conventional + additional electricity demand.

    Main options:
    - show_title=False removes the title from the figure.
    - y_label="Energy demand [kWh]" sets the vertical axis label.
    - x-axis dates are shown as DD-Mmm, without year.
    - save_pdf=True saves both PNG and PDF.
    """

    if column_order is None:
        column_order = list(df_plot.columns)

    if color_map is None:
        color_map = {}

    # Keep only available columns, preserving the desired order.
    columns_to_plot = [
        col for col in column_order
        if col in df_plot.columns
    ]

    # Add any unexpected columns at the end, instead of silently dropping them.
    columns_to_plot += [
        col for col in df_plot.columns
        if col not in columns_to_plot
    ]

    plot_df = (
        df_plot[columns_to_plot]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .clip(lower=0.0)
    )

    colors = [
        color_map.get(col, "#999999")
        for col in plot_df.columns
    ]

    with plt.rc_context({
        "font.size": font_size,
        "axes.labelsize": label_font_size,
        "xtick.labelsize": tick_font_size,
        "ytick.labelsize": tick_font_size,
        "legend.fontsize": legend_font_size,
        "axes.titlesize": title_font_size,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none"
    }):

        fig, ax = plt.subplots(figsize=fig_size)

        if isinstance(plot_df.index, pd.DatetimeIndex):
            x = plot_df.index

            ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
            ax.xaxis.set_major_formatter(plt.FuncFormatter(_format_day_month_en))
            ax.set_xlim(plot_df.index.min(), plot_df.index.max())

        else:
            x = np.arange(len(plot_df))

        ax.stackplot(
            x,
            [plot_df[col].to_numpy(dtype=float) for col in plot_df.columns],
            labels=plot_df.columns,
            colors=colors,
            alpha=alpha,
            linewidth=0.15,
            edgecolor="white"
        )

        ax.set_ylabel(y_label)
        ax.set_xlabel("")

        if show_title:
            ax.set_title(title, pad=8)

        ax.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.35)
        ax.grid(axis="x", visible=False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.tick_params(axis="x", rotation=0)
        ax.margins(x=0)

        if legend_position == "right":
            ax.legend(
                loc="upper left",
                bbox_to_anchor=(1.01, 1.00),
                frameon=False,
                handlelength=1.2,
                borderaxespad=0.0
            )

        elif legend_position == "bottom":
            ax.legend(
                loc="upper center",
                bbox_to_anchor=(0.5, -0.18),
                ncol=legend_ncol,
                frameon=False,
                handlelength=1.2,
                columnspacing=1.0,
                labelspacing=0.6
            )

        else:
            ax.legend(
                loc=legend_position,
                frameon=True,
                framealpha=0.95
            )

        os.makedirs(output_folder, exist_ok=True)

        output_path = os.path.join(output_folder, file_name)

        fig.savefig(
            output_path,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.04
        )

        if save_pdf:
            pdf_output_path = os.path.splitext(output_path)[0] + ".pdf"
            fig.savefig(
                pdf_output_path,
                bbox_inches="tight",
                pad_inches=0.04
            )

        plt.close(fig)

### 3.6.3 F(x) for combined daily activity + consumption plot

In [ ]:
#-------------------------------------------------------------------
#       Sec. 3.6 -- 3) FUNCTIONS FOR LINK ACTIVITY AND CONSUMPTIONS PLOT
#-------------------------------------------------------------------
# Function to return if daily plot has to be generated
def _season_selected_for_daily_plot(season, selected_seasons):
    """
    Return True if the combined daily plot should be generated for this season.
    """
    if selected_seasons is None:
        return False

    if selected_seasons == "all":
        return True

    if isinstance(selected_seasons, str):
        return season == selected_seasons

    return season in selected_seasons

# Function to return a copy of dataframe with a datetime index
def _ensure_datetime_index(df, datetime_col="Datetime"):
    """
    Return a copy of df with a DatetimeIndex.

    If df already has a DatetimeIndex, it is returned with a sorted index.
    If a datetime_col exists, it is converted to datetime and set as index.
    """
    out = df.copy()

    if isinstance(out.index, pd.DatetimeIndex):
        out = out.sort_index()
        return out

    if datetime_col not in out.columns:
        raise KeyError(
            f"Cannot create a DatetimeIndex: column '{datetime_col}' not found."
        )

    out[datetime_col] = pd.to_datetime(out[datetime_col])
    out = out.set_index(datetime_col).sort_index()

    return out


# Function to extract a day profile from the datetime index df
def _extract_one_day(
    df,
    selected_date=None,
    day_index=0,
    expected_steps=144,
    label="dataframe"
):
    """
    Extract one complete day from a DatetimeIndex DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with DatetimeIndex, or with a Datetime column already
        converted before calling this function.
    selected_date : str or pandas-compatible datetime, optional
        Explicit day to extract, e.g. "2023-02-15".
        If None, day_index is used.
    day_index : int
        Position of the day within the available weekly profile.
        0 means first day.
    expected_steps : int
        Expected number of 10-minute timesteps in one day.
    label : str
        Label used only in warning messages.

    Returns
    -------
    tuple[pd.DataFrame, pd.Timestamp]
        Daily DataFrame and selected day as normalized Timestamp.
    """
    df = _ensure_datetime_index(df)

    available_days = pd.Index(df.index.normalize().unique()).sort_values()

    if len(available_days) == 0:
        raise ValueError(f"No days found in {label}.")

    if selected_date is None:
        if day_index < 0 or day_index >= len(available_days):
            raise IndexError(
                f"day_index={day_index} is out of range for {label}. "
                f"Available days: {list(available_days)}"
            )

        selected_day = pd.Timestamp(available_days[day_index]).normalize()

    else:
        selected_day = pd.Timestamp(selected_date).normalize()

        if selected_day not in available_days:
            raise ValueError(
                f"Selected date {selected_day.date()} not found in {label}. "
                f"Available days: {[pd.Timestamp(d).date() for d in available_days]}"
            )

    daily_df = df.loc[df.index.normalize() == selected_day].copy()

    if len(daily_df) != expected_steps:
        print(
            f"[WARN][DAILY PLOT] {label}: selected day {selected_day.date()} "
            f"has {len(daily_df)} timesteps instead of {expected_steps}."
        )

    return daily_df, selected_day


# Function to build an aggregated activity distribution dataframe
def build_daily_activity_distribution(
    activity_day_df,
    activity_code_to_label,
    categories
):
    """
    Convert a person-level activity-code DataFrame into an aggregated activity
    distribution for one day.

    Input:
        rows    = 10-minute timesteps
        columns = household members
        values  = numerical activity codes

    Output:
        rows    = same 10-minute timesteps
        columns = activity categories
        values  = number of household members performing each activity
    """
    activity_values = activity_day_df.copy()

    # Remove a possible Datetime column if the dataframe was not indexed yet.
    activity_values = activity_values.drop(columns=["Datetime"], errors="ignore")

    # Keep only numerical activity-code values.
    activity_values = activity_values.apply(pd.to_numeric, errors="coerce")

    activity_distribution_df = pd.DataFrame(index=activity_values.index)

    for activity_code, activity_label in activity_code_to_label.items():
        activity_distribution_df[activity_label] = (
            activity_values.eq(activity_code).sum(axis=1).astype(float)
        )

    activity_distribution_df = (
        activity_distribution_df
        .reindex(columns=categories, fill_value=0.0)
        .fillna(0.0)
    )

    return activity_distribution_df


# Function to define the format for axes in graphs
def _format_hour_axis(ax, x_index, tick_interval_hours=2):
    """
    Format a shared x-axis as hour of the day.
    """
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=tick_interval_hours))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    if len(x_index) > 0:
        ax.set_xlim(x_index.min(), x_index.max())


# Function to return a plot with aggregated activity profiles and related consumption of a tested n° of households
def plot_daily_activity_and_consumption(
        activity_week_df,
        consumption_week_df,
        title,
        file_name,
        output_folder,
        selected_date=None,
        day_index=0,
        activity_categories=None,
        activity_code_to_label=None,
        activity_english_dict=None,
        activity_color_map=None,
        consumption_column_order=None,
        consumption_color_map=None,
        expected_steps_per_day=144,
        show_title=False,
        fig_size=(7.20, 5.80),
        font_size=9.0,
        label_font_size=10.0,
        tick_font_size=8.5,
        activity_legend_font_size=6.2,
        consumption_legend_font_size=8.0,
        title_font_size=11.0,
        activity_alpha=0.90,
        consumption_alpha=0.95,
        save_pdf=True):
    """
    Create a two-panel daily plot with a shared x-axis.

    Upper panel:
        aggregated activity distribution across household members.

    Lower panel:
        stacked conventional + additional energy demand.

    The function does not replace plot_stacked_consumption_for_paper; it is an
    optional additional plot for comparing activities and loads on the same day.
    """
    if activity_categories is None:
        activity_categories = aggregated_activities

    if activity_code_to_label is None:
        activity_code_to_label = globals()["activity_code_to_label"]

    if activity_english_dict is None:
        activity_english_dict = globals()["activity_english_dict"]

    if activity_color_map is None:
        activity_color_map = globals()["activity_color_map"]

    if consumption_column_order is None:
        consumption_column_order = list(consumption_week_df.columns)

    if consumption_color_map is None:
        consumption_color_map = {}

    # 1) Extract the same selected day from activity and consumption profiles.
    activity_day_df, selected_day = _extract_one_day(
        activity_week_df,
        selected_date=selected_date,
        day_index=day_index,
        expected_steps=expected_steps_per_day,
        label="activity profile"
    )

    consumption_day_df, _ = _extract_one_day(
        consumption_week_df,
        selected_date=selected_day,
        day_index=0,
        expected_steps=expected_steps_per_day,
        label="consumption profile"
    )

    # 2) Build the upper-panel activity distribution.
    activity_distribution_df = build_daily_activity_distribution(
        activity_day_df=activity_day_df,
        activity_code_to_label=activity_code_to_label,
        categories=activity_categories
    )

    # 3) Prepare the lower-panel consumption dataframe.
    columns_to_plot = [
        col for col in consumption_column_order
        if col in consumption_day_df.columns
    ]

    columns_to_plot += [
        col for col in consumption_day_df.columns
        if col not in columns_to_plot
    ]

    consumption_day_df = (
        consumption_day_df[columns_to_plot]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .clip(lower=0.0)
    )

    # If indexes do not match but lengths do, align by position.
    # This is useful when one input was read from Excel and lost exact datetime metadata.
    if not consumption_day_df.index.equals(activity_distribution_df.index):
        if len(consumption_day_df) == len(activity_distribution_df):
            consumption_day_df.index = activity_distribution_df.index
        else:
            consumption_day_df = (
                consumption_day_df
                .reindex(activity_distribution_df.index)
                .fillna(0.0)
            )

    x = activity_distribution_df.index

    with plt.rc_context({
        "font.size": font_size,
        "axes.labelsize": label_font_size,
        "xtick.labelsize": tick_font_size,
        "ytick.labelsize": tick_font_size,
        "axes.titlesize": title_font_size,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none"
    }):

        fig, (ax_activity, ax_consumption) = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=fig_size,
            sharex=True,
            gridspec_kw={
                "height_ratios": [1.05, 1.00],
                "hspace": 0.08
            }
        )

        # -----------------------------
        # Upper panel: activity profiles
        # -----------------------------
        activity_bottom = np.zeros(len(activity_distribution_df), dtype=float)

        for activity_label in activity_categories:
            if activity_label not in activity_distribution_df.columns:
                continue

            values = activity_distribution_df[activity_label].to_numpy(dtype=float)

            if np.isclose(values.sum(), 0.0):
                continue

            color = activity_color_map.get(activity_label, "#999999")
            legend_label = activity_english_dict.get(activity_label, activity_label)

            ax_activity.fill_between(
                x,
                activity_bottom,
                activity_bottom + values,
                label=legend_label,
                color=color,
                alpha=activity_alpha,
                linewidth=0.15,
                edgecolor="white"
            )

            activity_bottom = activity_bottom + values

        ax_activity.set_ylabel("Number of people")

        if len(activity_bottom) > 0 and activity_bottom.max() > 0:
            ax_activity.set_ylim(0, activity_bottom.max() * 1.03)

        ax_activity.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.40)
        ax_activity.grid(axis="x", visible=False)
        ax_activity.spines["top"].set_visible(False)
        ax_activity.spines["right"].set_visible(False)

        activity_handles, activity_labels = ax_activity.get_legend_handles_labels()
        ax_activity.legend(
            activity_handles,
            activity_labels,
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            frameon=False,
            fontsize=activity_legend_font_size,
            handlelength=1.2,
            borderaxespad=0.0
        )

        # ---------------------------------------
        # Lower panel: conventional + additional
        # ---------------------------------------
        consumption_colors = [
            consumption_color_map.get(col, "#999999")
            for col in consumption_day_df.columns
        ]

        ax_consumption.stackplot(
            x,
            [consumption_day_df[col].to_numpy(dtype=float) for col in consumption_day_df.columns],
            labels=consumption_day_df.columns,
            colors=consumption_colors,
            alpha=consumption_alpha,
            linewidth=0.15,
            edgecolor="white"
        )

        ax_consumption.set_ylabel("Energy demand [kWh]")
        ax_consumption.set_xlabel("Hour of the day")

        ax_consumption.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.35)
        ax_consumption.grid(axis="x", visible=False)
        ax_consumption.spines["top"].set_visible(False)
        ax_consumption.spines["right"].set_visible(False)

        consumption_handles, consumption_labels = ax_consumption.get_legend_handles_labels()
        ax_consumption.legend(
            consumption_handles,
            consumption_labels,
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            frameon=False,
            fontsize=consumption_legend_font_size,
            handlelength=1.2,
            borderaxespad=0.0
        )

        _format_hour_axis(ax_consumption, x, tick_interval_hours=2)

        if show_title:
            fig.suptitle(title, y=0.995, fontsize=title_font_size)

        os.makedirs(output_folder, exist_ok=True)

        output_path = os.path.join(output_folder, file_name)

        fig.savefig(
            output_path,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.04
        )

        if save_pdf:
            pdf_output_path = os.path.splitext(output_path)[0] + ".pdf"
            fig.savefig(
                pdf_output_path,
                bbox_inches="tight",
                pad_inches=0.04
            )

        plt.close(fig)

    return output_path



# -------------------------------------------------------
#   Sec 3.6 -- 3.2) Functions for percentile
# -------------------------------------------------------
# Save a Matplotlib figure as PNG and optionally as PDF.
def save_plot_figure(
        fig,
        output_path,
        save_pdf=False,
        dpi=300):

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True,
    )

    fig.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.04,
    )

    if save_pdf:
        fig.savefig(
            os.path.splitext(output_path)[0] + ".pdf",
            bbox_inches="tight",
            pad_inches=0.04,
        )


# Plot annual percentile bands across all households for one metric.
def plot_annual_percentiles_all_households(
        series,
        metric_label,
        metric_unit,
        file_name,
        output_folder,
        percentile_levels,
        percentile_bands,
        show_title=True,
        save_pdf=False):

    values = pd.to_numeric(
        series,
        errors="coerce",
    ).dropna()

    if values.empty:
        print(
            f"[SKIP][PERCENTILES] No valid values for "
            f"'{metric_label}'."
        )
        return None

    percentile_values = np.percentile(
        values.to_numpy(dtype=float),
        percentile_levels,
    )

    percentile_map = dict(
        zip(
            percentile_levels,
            percentile_values,
        )
    )

    mean_value = float(values.mean())
    median_value = float(values.median())

    color_map = plt.get_cmap("Blues")

    band_colors = [
        color_map(value)
        for value in np.linspace(
            0.15,
            0.75,
            len(percentile_bands),
        )
    ]

    fig, ax = plt.subplots(
        figsize=(5.0, 7.0),
        dpi=150,
    )

    x_position = np.array([0])
    bar_width = 0.60

    for (
        low_percentile,
        high_percentile,
    ), color in zip(
        percentile_bands,
        band_colors,
    ):
        low_value = percentile_map[low_percentile]
        high_value = percentile_map[high_percentile]

        ax.bar(
            x_position,
            high_value - low_value,
            bottom=low_value,
            width=bar_width,
            color=color,
            alpha=0.95,
            edgecolor="none",
        )

    ax.axhline(
        median_value,
        color="black",
        linestyle="--",
        linewidth=1.6,
    )

    ax.axhline(
        mean_value,
        color="red",
        linestyle="-.",
        linewidth=1.4,
    )

    ax.set_xticks(x_position)
    ax.set_xticklabels([metric_label])
    ax.set_ylabel(f"{metric_label} [{metric_unit}]")

    if show_title:
        ax.set_title(
            f"Annual distribution — {metric_label}\n"
            f"{num_households} households"
        )

    ax.grid(
        axis="y",
        linestyle=":",
        alpha=0.35,
    )

    for percentile in (10, 50, 90):
        percentile_value = percentile_map[percentile]

        ax.text(
            x_position[0] + bar_width / 2 + 0.05,
            percentile_value,
            f"p{percentile}={percentile_value:.1f}",
            va="center",
            fontsize=9,
        )

    legend_handles = [
        Patch(
            facecolor=color,
            label=f"{low}-{high}%",
        )
        for (low, high), color in zip(
            percentile_bands,
            band_colors,
        )
    ]

    legend_handles.extend(
        [
            Line2D(
                [0],
                [0],
                color="black",
                linestyle="--",
                label="Median",
            ),
            Line2D(
                [0],
                [0],
                color="red",
                linestyle="-.",
                label="Mean",
            ),
        ]
    )

    ax.legend(
        handles=legend_handles,
        loc="best",
    )

    fig.tight_layout()

    output_path = os.path.join(
        output_folder,
        file_name,
    )

    save_plot_figure(
        fig,
        output_path,
        save_pdf=save_pdf,
        dpi=300,
    )

    plt.close(fig)

    return output_path


# Plot annual percentile bands by household type for one metric.
def plot_annual_percentiles_by_household_type(
        annual_df,
        value_column,
        metric_label,
        metric_unit,
        file_name,
        output_folder,
        percentile_levels,
        percentile_bands,
        household_type_column="Household type",
        show_title=True,
        save_pdf=False):

    if household_type_column not in annual_df.columns:
        print(
            f"[SKIP][PERCENTILES] Column "
            f"'{household_type_column}' not found."
        )
        return None

    household_types = sorted(
        annual_df[household_type_column]
        .dropna()
        .unique()
        .tolist()
    )

    percentile_values_by_level = {
        percentile: []
        for percentile in percentile_levels
    }

    means = []
    counts = []

    for household_type in household_types:
        values = pd.to_numeric(
            annual_df.loc[
                annual_df[household_type_column]
                == household_type,
                value_column,
            ],
            errors="coerce",
        ).dropna()

        if values.empty:
            for percentile in percentile_levels:
                percentile_values_by_level[
                    percentile
                ].append(np.nan)

            means.append(np.nan)
            counts.append(0)
            continue

        percentiles = np.percentile(
            values.to_numpy(dtype=float),
            percentile_levels,
        )

        for percentile, value in zip(
            percentile_levels,
            percentiles,
        ):
            percentile_values_by_level[
                percentile
            ].append(value)

        means.append(float(values.mean()))
        counts.append(int(len(values)))

    percentile_arrays = {
        percentile: np.asarray(
            values,
            dtype=float,
        )
        for percentile, values
        in percentile_values_by_level.items()
    }

    means = np.asarray(
        means,
        dtype=float,
    )

    x_positions = np.arange(
        len(household_types)
    )

    bar_width = 0.60

    color_map = plt.get_cmap("Blues")

    band_colors = [
        color_map(value)
        for value in np.linspace(
            0.15,
            0.75,
            len(percentile_bands),
        )
    ]

    fig, ax = plt.subplots(
        figsize=(
            max(
                10.0,
                len(household_types) * 1.4,
            ),
            6.0,
        ),
        dpi=150,
    )

    for (
        low_percentile,
        high_percentile,
    ), color in zip(
        percentile_bands,
        band_colors,
    ):
        low_values = percentile_arrays[
            low_percentile
        ]

        high_values = percentile_arrays[
            high_percentile
        ]

        low_plot = np.nan_to_num(
            low_values,
            nan=0.0,
        )

        height_plot = np.nan_to_num(
            high_values - low_values,
            nan=0.0,
        )

        ax.bar(
            x_positions,
            height_plot,
            bottom=low_plot,
            width=bar_width,
            color=color,
            alpha=0.95,
            edgecolor="none",
        )

    ax.scatter(
        x_positions,
        percentile_arrays[50],
        marker="o",
        color="black",
        zorder=10,
        label="Median",
    )

    ax.scatter(
        x_positions,
        means,
        marker="x",
        color="red",
        zorder=11,
        label="Mean",
    )

    ax.set_xticks(x_positions)

    ax.set_xticklabels(
        household_types,
        rotation=30,
        ha="right",
    )

    ax.set_ylabel(
        f"{metric_label} [{metric_unit}]"
    )

    ax.set_xlabel(
        "Household type"
    )

    if show_title:
        ax.set_title(
            f"Annual distribution by household type — "
            f"{metric_label}\n"
            f"{num_households} households"
        )

    ax.grid(
        axis="y",
        linestyle=":",
        alpha=0.35,
    )

    for x_position, count in zip(
        x_positions,
        counts,
    ):
        ax.text(
            x_position,
            -0.20,
            f"n={count}",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=8,
        )

    legend_handles = [
        Patch(
            facecolor=color,
            label=f"{low}-{high}%",
        )
        for (low, high), color in zip(
            percentile_bands,
            band_colors,
        )
    ]

    legend_handles.extend(
        [
            Line2D(
                [0],
                [0],
                color="black",
                marker="o",
                linestyle="None",
                label="Median",
            ),
            Line2D(
                [0],
                [0],
                color="red",
                marker="x",
                linestyle="None",
                label="Mean",
            ),
        ]
    )

    ax.legend(
        handles=legend_handles,
        loc="best",
    )

    fig.subplots_adjust(
        bottom=0.28,
    )

    output_path = os.path.join(
        output_folder,
        file_name,
    )

    save_plot_figure(
        fig,
        output_path,
        save_pdf=save_pdf,
        dpi=300,
    )

    plt.close(fig)

    return output_path


# Function to calculate annual descriptive statistics overall and by household type.
def build_annual_statistics_tables(
        annual_df,
        metric_specs,
        percentile_levels,
        household_type_column="Household type"):

    statistic_columns = (
        ["count", "mean", "median", "std", "min"]
        + [f"p{percentile}" for percentile in percentile_levels]
        + ["max"]
    )

    def describe_series(series):
        values = pd.to_numeric(
            series,
            errors="coerce",
        ).dropna()

        if values.empty:
            return {
                column: np.nan
                for column in statistic_columns
            }

        result = {
            "count": int(values.size),
            "mean": float(values.mean()),
            "median": float(values.median()),
            "std": float(values.std()),
            "min": float(values.min()),
            "max": float(values.max()),
        }

        percentile_values = np.percentile(
            values.to_numpy(dtype=float),
            percentile_levels,
        )

        for percentile, value in zip(
            percentile_levels,
            percentile_values,
        ):
            result[f"p{percentile}"] = float(value)

        return result

    overall_rows = []
    household_type_rows = []

    for value_column, metric_spec in metric_specs.items():

        if value_column not in annual_df.columns:
            continue

        overall_row = {
            "metric_column": value_column,
            "metric": metric_spec["label"],
            "unit": metric_spec["unit"],
        }

        overall_row.update(
            describe_series(
                annual_df[value_column]
            )
        )

        overall_rows.append(overall_row)

        if household_type_column not in annual_df.columns:
            continue

        for household_type, group_df in annual_df.groupby(
            household_type_column,
            dropna=False,
        ):
            household_type_row = {
                "metric_column": value_column,
                "metric": metric_spec["label"],
                "unit": metric_spec["unit"],
                "household_type": household_type,
            }

            household_type_row.update(
                describe_series(
                    group_df[value_column]
                )
            )

            household_type_rows.append(
                household_type_row
            )

    overall_statistics_df = pd.DataFrame(
        overall_rows
    )

    household_type_statistics_df = pd.DataFrame(
        household_type_rows
    )

    return (
        overall_statistics_df,
        household_type_statistics_df,
    )

### 3.6.4 F(x) for additional-load definition

In [ ]:

#-------------------------------------------------------------------
#       Sec. 3.6 -- 4.1) FUNCTIONS FOR DEFINING HOUSEHOLD STRUCTURE
#-------------------------------------------------------------------
# The following functions identify which household types are present,
# which members belong to each household, and how individual-level
# quantities should be aggregated at household level.

# Function to identify the household associated with each activity-profile column.
def identify_households(activity_df):
    """
    Identify households and household types from the column names of the
    activity-profile DataFrame.

    Expected column format:
        "Household_(number)_Member_(number)_(season)_(household_type)"

    Returns
    -------
    tuple[pd.Series, pd.Series]
        - household identifiers, e.g. "Household_1";
        - canonical English household types.
    """
    extracted = activity_df.columns.to_series().str.extract(
        r'(Household_\d+)_Member_\d+_[^_]+_(.*)'
    )
    extracted.columns = ['Household', 'Household type']

    if extracted.isnull().any().any():
        raise ValueError(
            "Invalid column format in at least one activity-profile column."
        )

    return extracted['Household'], extracted['Household type'].map(canonical_household_type)


# Function to infer whether a household member is at home.
def infer_occupancy_hybrid_probabilistic(series: pd.Series, lookahead: int = 6) -> pd.Series:
    """
    Return a 0/1 occupancy flag for each timestep.

    Logic:
    - travel blocks, activity code 7, are always treated as away from home;
    - before the first travel block and after the last one, occupancy is inferred
        directly from the activity category at each timestep;
    - between two travel blocks, a single occupancy state is sampled for the whole
        interval using the average p_home value of the activities in that interval;
    - for selected activities, such as Eating and Phone use, p_home can depend on
        nearby Work activities.
    """
    vals = series.values
    n = len(vals)
    occ = np.full(n, np.nan, dtype=float)

    # Identify travel blocks.
    is_travel = vals == 7
    diffs = np.diff(is_travel.astype(int), prepend=0, append=0)
    starts = np.nonzero(diffs == 1)[0]
    ends = np.nonzero(diffs == -1)[0] - 1

    # 1) Before the first travel block: timestep-by-timestep logic.
    first_start = starts[0] if starts.size > 0 else n
    for i in range(0, first_start):
        code_i = vals[i]
        cat = activity_map.get(code_i, {'cat': 'maybe'})['cat']

        if cat == 'home':
            occ[i] = 1
        elif cat == 'away':
            occ[i] = 0
        else:
            occ[i] = occ[i - 1] if i > 0 else 1

    # 2) Travel blocks: always away from home.
    for block_start, block_end in zip(starts, ends):
        occ[block_start:block_end + 1] = 0

    # 3) Intervals between travel blocks: assign one occupancy state per interval.
    for idx in range(len(starts) - 1):
        seg_start = ends[idx] + 1
        seg_end = starts[idx + 1] - 1

        if seg_start > seg_end:
            continue

        p_home_values = []

        for j in range(seg_start, seg_end + 1):
            code_j = vals[j]

            # For Eating (3) and Phone use (22), check whether Work (14)
            # occurs in the following lookahead timesteps without an intervening
            # travel event.
            if code_j in (3, 22):
                future = vals[j + 1:min(n, j + 1 + lookahead)]
                p_home_j = 1.0

                if 14 in future:
                    rel = list(future).index(14)
                    abs_work_idx = j + 1 + rel

                    if 7 not in vals[j + 1:abs_work_idx]:
                        p_home_j = activity_map[14]['p_home']
            else:
                p_home_j = activity_map.get(code_j, {'p_home': 0.5})['p_home']

            p_home_values.append(p_home_j)

        interval_probability = sum(p_home_values) / len(p_home_values)

        # One Bernoulli draw for the whole interval.
        interval_state = np.random.binomial(1, interval_probability)
        occ[seg_start:seg_end + 1] = interval_state

    # 4) After the last travel block: timestep-by-timestep logic.
    if ends.size > 0 and ends[-1] < n - 1:
        for i in range(ends[-1] + 1, n):
            code_i = vals[i]
            cat = activity_map.get(code_i, {'cat': 'maybe'})['cat']

            if cat == 'home':
                occ[i] = 1
            elif cat == 'away':
                occ[i] = 0
            else:
                occ[i] = occ[i - 1]

    # If no travel blocks are present, the first block already filled the series.
    # Fill any remaining NaN values for safety.
    occ = np.where(np.isnan(occ), 1, occ).astype(int)

    return pd.Series(occ, index=series.index)


# Function to extract the household identifier from a person-level column name.
def person_to_household(col_name):
    match = re.match(r'^(Household_\d+)_', col_name)
    return match.group(1) if match else None


# Function to aggregate person-level values to household-level values.
def aggregate_person_to_household_sum(person_series):
    household_values = {}

    for person, value in person_series.items():
        household_id = person_to_household(person)

        if household_id is None:
            continue

        household_values.setdefault(household_id, 0.0)
        household_values[household_id] += float(value)

    return pd.Series(household_values).sort_index()



#-------------------------------------------------------------------
#       Sec. 3.6 -- 4.2) FUNCTIONS FOR DEFINING ADDITIONAL LOADS
#-------------------------------------------------------------------

# --------
#  COOKING

# Function to calculate induction-hob electricity consumption.
def calculate_cooking_consumption_by_season(
    activity_df: pd.DataFrame,
    induction_ownership: dict,
    induction_use_probability: float,
    induction_consumption_kwh: float
) -> pd.DataFrame:
    """
    Return a person-level DataFrame of induction-hob electricity consumption.

    Parameters
    ----------
    activity_df : pd.DataFrame
        Activity-profile DataFrame with one column per household member.
    induction_ownership : dict
        Household-level dictionary indicating whether each household owns an
        induction hob.
    induction_use_probability : float
        Probability of using the induction hob when activity code 6, Cooking,
        occurs.
    induction_consumption_kwh : float
        Electricity consumption assigned to each cooking event.
    """
    df_cooking = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)

    for person in activity_df.columns:
        simple_household_id = re.match(r'(Household_\d+)', person).group(1)

        if not induction_ownership.get(simple_household_id, False):
            continue

        for ts, activity_code in activity_df[person].items():
            if activity_code == 6 and random.random() < induction_use_probability:
                df_cooking.at[ts, person] = induction_consumption_kwh

    return df_cooking


# Function to avoid or down-weight overlapping cooking events within households.
def resolve_cooking_overlap_by_household(
    df_cook_person: pd.DataFrame,
    window_ts: int = 2,
    secondary_event_weight: float = 0.0,
    main_event_criterion: str = "max_energy"   # "max_energy" or "first"
) -> pd.DataFrame:
    """
    Post-process cooking consumption to avoid double counting among members
    of the same household.

    Logic:
    1) consecutive positive timesteps for each person are grouped into one
        cooking event;
    2) events close in time within the same household are grouped into clusters;
    3) within each cluster, one main event is kept at 100%, while the remaining
        events are either removed or down-weighted.
    """
    if df_cook_person.empty:
        return df_cook_person.copy()

    df_out = pd.DataFrame(
        0.0,
        index=df_cook_person.index,
        columns=df_cook_person.columns,
        dtype=float
    )

    # Map each person-level column to the corresponding household.
    household_of_person = df_cook_person.columns.to_series().str.extract(r'^(Household_\d+)')[0]

    for household_id in household_of_person.dropna().unique():
        cols = household_of_person[household_of_person == household_id].index.tolist()
        events = []

        # Build person-level cooking events as consecutive positive-consumption blocks.
        for person in cols:
            vals = df_cook_person[person].to_numpy(dtype=float)
            active = vals > 0

            if not active.any():
                continue

            starts = np.flatnonzero(active & np.r_[True, ~active[:-1]])
            ends = np.flatnonzero(active & np.r_[~active[1:], True])

            for start, end in zip(starts, ends):
                events.append({
                    "person": person,
                    "start": int(start),
                    "end": int(end),
                    "energy": float(vals[start:end + 1].sum())
                })

        if not events:
            continue

        # Sort events and create time-close clusters.
        events.sort(key=lambda x: x["start"])

        clusters = []
        current_cluster = [events[0]]
        current_max_end = events[0]["end"]

        for event in events[1:]:
            # Same cluster if the event starts within window_ts from the
            # end of the active cluster.
            if event["start"] <= current_max_end + window_ts:
                current_cluster.append(event)
                current_max_end = max(current_max_end, event["end"])
            else:
                clusters.append(current_cluster)
                current_cluster = [event]
                current_max_end = event["end"]

        clusters.append(current_cluster)

        # Within each cluster, keep one main event.
        for cluster in clusters:
            if main_event_criterion == "first":
                main_event = min(cluster, key=lambda x: (x["start"], x["end"]))
            else:
                main_event = max(cluster, key=lambda x: (x["energy"], -x["start"]))

            for event in cluster:
                person = event["person"]
                start, end = event["start"], event["end"]

                src_col = df_cook_person.columns.get_loc(person)
                dst_col = df_out.columns.get_loc(person)

                segment = df_cook_person.iloc[start:end + 1, src_col].to_numpy(dtype=float)

                weight = 1.0 if event is main_event else float(secondary_event_weight)

                if weight > 0:
                    df_out.iloc[start:end + 1, dst_col] += weight * segment

    return df_out


# Function to diagnose cooking overlaps within households.
def diagnose_cooking_overlap_by_household(
    df_cook_person: pd.DataFrame,
    window_ts: int = 2
):
    """
    Diagnose cooking overlap within each household.

    Returns
    -------
    tuple
        - df_ts_events: timestep-level overlap events, where at least two members
            cook in the same timestep;
        - timestep_count_by_household: number of overlapping timesteps by household;
        - df_cluster_events: time-close cooking clusters involving at least two
            members of the same household;
        - cluster_count_by_household: number of overlapping clusters by household;
        - df_summary: compact household-level summary.
    """

    if df_cook_person.empty:
        empty_df = pd.DataFrame()
        empty_s = pd.Series(dtype=float)
        return empty_df, empty_s, empty_df, empty_s, empty_df

    # A) Timestep-level simultaneous-overlap diagnostics.
    cook_bin = (df_cook_person > 0).astype(int)
    household_of_person = cook_bin.columns.to_series().str.extract(r'^(Household_\d+)')[0]

    cooking_count_by_household = cook_bin.T.groupby(household_of_person).sum().T
    overlap_mask_ts = cooking_count_by_household >= 2

    ts_events = overlap_mask_ts.stack()
    ts_events = ts_events[ts_events]

    if len(ts_events) > 0:
        df_ts_events = ts_events.index.to_frame(index=False)
        df_ts_events.columns = ["Datetime", "Household"]
        df_ts_events["n_members_cooking"] = (
            cooking_count_by_household.stack().reindex(ts_events.index).values
        )
    else:
        df_ts_events = pd.DataFrame(columns=["Datetime", "Household", "n_members_cooking"])

    timestep_count_by_household = overlap_mask_ts.sum(axis=0).rename("n_timesteps_overlap")

    # B) Event-level and cluster-level diagnostics.
    all_cluster_rows = []
    cluster_count_dict = {}

    for household_id in household_of_person.dropna().unique():
        cols = household_of_person[household_of_person == household_id].index.tolist()
        events = []

        # Reconstruct person-level cooking events as consecutive positive blocks.
        for person in cols:
            vals = df_cook_person[person].to_numpy(dtype=float)
            active = vals > 0

            if not active.any():
                continue

            starts = np.flatnonzero(active & np.r_[True, ~active[:-1]])
            ends = np.flatnonzero(active & np.r_[~active[1:], True])

            for start, end in zip(starts, ends):
                events.append({
                    "Household": household_id,
                    "person": person,
                    "start_idx": int(start),
                    "end_idx": int(end),
                    "start_time": df_cook_person.index[start],
                    "end_time": df_cook_person.index[end],
                    "duration_ts": int(end - start + 1),
                    "energy_kWh": float(vals[start:end + 1].sum())
                })

        if not events:
            cluster_count_dict[household_id] = 0
            continue

        events = sorted(events, key=lambda x: x["start_idx"])

        clusters = []
        current_cluster = [events[0]]
        current_end = events[0]["end_idx"]

        for event in events[1:]:
            if event["start_idx"] <= current_end + window_ts:
                current_cluster.append(event)
                current_end = max(current_end, event["end_idx"])
            else:
                clusters.append(current_cluster)
                current_cluster = [event]
                current_end = event["end_idx"]

        clusters.append(current_cluster)

        # Keep only clusters involving at least two different people.
        n_overlap_clusters = 0

        for i, cluster in enumerate(clusters, start=1):
            people_involved = sorted(set(event["person"] for event in cluster))

            if len(people_involved) < 2:
                continue

            n_overlap_clusters += 1

            start_idx_cluster = min(event["start_idx"] for event in cluster)
            end_idx_cluster = max(event["end_idx"] for event in cluster)

            all_cluster_rows.append({
                "Household": household_id,
                "cluster_id": f"{household_id}_cluster_{i}",
                "start_cluster": df_cook_person.index[start_idx_cluster],
                "end_cluster": df_cook_person.index[end_idx_cluster],
                "cluster_duration_ts": int(end_idx_cluster - start_idx_cluster + 1),
                "n_events_cluster": int(len(cluster)),
                "n_people_involved": int(len(people_involved)),
                "people_involved": " | ".join(people_involved),
                "cluster_energy_kWh": float(sum(event["energy_kWh"] for event in cluster))
            })

        cluster_count_dict[household_id] = n_overlap_clusters

    df_cluster_events = pd.DataFrame(all_cluster_rows)
    cluster_count_by_household = pd.Series(cluster_count_dict).rename("n_cluster_overlap")

    # C) Household-level summary.
    idx = sorted(set(timestep_count_by_household.index).union(set(cluster_count_by_household.index)))
    df_summary = pd.DataFrame(index=idx)
    df_summary["n_timesteps_overlap"] = timestep_count_by_household.reindex(idx).fillna(0).astype(int)
    df_summary["n_cluster_overlap"] = cluster_count_by_household.reindex(idx).fillna(0).astype(int)

    if not cooking_count_by_household.empty:
        df_summary["max_simultaneous_members_per_ts"] = (
            cooking_count_by_household.max(axis=0).reindex(idx).fillna(0).astype(int)
        )
    else:
        df_summary["max_simultaneous_members_per_ts"] = 0

    return df_ts_events, timestep_count_by_household, df_cluster_events, cluster_count_by_household, df_summary


# --------
#  DHW

# Function to calculate domestic hot water demand associated with showers.
def calculate_dhw_consumption_by_season(
    activity_df: pd.DataFrame,
    shower_ownership: dict,
    morning_shower_use_probability: float,
    daytime_shower_use_probability: float,
    evening_shower_use_probability: float,
    min_consumption: float,
    max_consumption: float,
    eta_boiler: float,
    personal_care_cluster_max_gap_ts: int = 2,
    household_type_by_household: dict = None,
):
    """
    Generate person-level DHW demand associated with showers.

    Main rules
    ----------
    1. Nearby occurrences of activity code 2 (Personal care & dressing)
       are grouped into one personal-care EVENT.
    2. One Bernoulli trial is performed per personal-care event,
       not per timestep.
    3. Morning and afternoon probabilities can differ.
    4. Maximum one shower per person per calendar day.
    5. Shower energy is sampled ONCE per accepted shower and represents
       the total thermal energy of that shower.
    6. The original activity profile is never modified.

    Returns
    -------
    dhw_df
        Person-level RAW shower-demand dataframe [kWh_th/event].

    event_audit_df
        One row for every reconstructed personal-care event, including
        whether it generated a RAW shower.
    """

    dhw_df = pd.DataFrame(
        0.0,
        index=activity_df.index,
        columns=activity_df.columns,
        dtype=float,
    )

    max_gap_ts = max(
        1,
        int(personal_care_cluster_max_gap_ts),
    )

    # -------------------------------------------------------------
    # Household membership and REAL family size.
    #
    # Important: family size is derived from the actual number of
    # person columns, NOT from equivalent_people in dhw_specs.
    # This avoids treating 2-person single-parent households as size 3.
    # -------------------------------------------------------------
    household_of_person = {}

    for person in activity_df.columns:
        match = re.match(
            r"^(Household_\d+)",
            str(person),
        )

        if match is not None:
            household_of_person[person] = match.group(1)

    family_size_by_household = {}

    for household_id in set(household_of_person.values()):
        family_size_by_household[household_id] = sum(
            1
            for p, h in household_of_person.items()
            if h == household_id
        )

    audit_rows = []
    event_counter = 0

    # -------------------------------------------------------------
    # Person-by-person reconstruction
    # -------------------------------------------------------------
    for person in activity_df.columns:

        household_id = household_of_person.get(person)

        if household_id is None:
            continue

        if not shower_ownership.get(
            household_id,
            False,
        ):
            continue

        activity_values = (
            activity_df[person]
            .to_numpy()
        )

        personal_care_positions = np.flatnonzero(
            activity_values == 2
        )

        if len(personal_care_positions) == 0:
            continue

        # ---------------------------------------------------------
        # Build Personal-care clusters.
        #
        # Example with max_gap_ts = 2:
        #
        #   PC - PC - Breakfast - PC
        #
        # becomes one event.
        #
        # The Breakfast timestep remains untouched in activity_df.
        # ---------------------------------------------------------
        clusters = []

        current_cluster = [
            int(personal_care_positions[0])
        ]

        for pos_raw in personal_care_positions[1:]:

            pos = int(pos_raw)
            previous_pos = current_cluster[-1]

            same_day = (
                activity_df.index[pos].date()
                ==
                activity_df.index[previous_pos].date()
            )

            close_enough = (
                pos - previous_pos
                <= max_gap_ts
            )

            if same_day and close_enough:
                current_cluster.append(pos)

            else:
                clusters.append(current_cluster)
                current_cluster = [pos]

        clusters.append(current_cluster)

        # ---------------------------------------------------------
        # Number of Personal-care EVENTS for this person on each day.
        # This is more useful than merely counting activity-code
        # timesteps.
        # ---------------------------------------------------------
        cluster_day = {
            i: activity_df.index[cluster[0]].date()
            for i, cluster in enumerate(clusters)
        }

        n_events_by_day = (
            pd.Series(list(cluster_day.values()))
            .value_counts()
            .to_dict()
        )

        # Maximum one accepted shower per person/day.
        shower_dates = set()

        for cluster_idx, cluster in enumerate(clusters):

            event_counter += 1

            event_id = (
                f"DHW_{event_counter:06d}"
            )

            start_pos = cluster[0]
            end_pos = cluster[-1]

            start_ts = activity_df.index[start_pos]
            end_ts = activity_df.index[end_pos]

            day = start_ts.date()

            n_events_same_day = int(
                n_events_by_day.get(day, 1)
            )



            hour = start_ts.hour

            # ---------------------------------------------------------
            # Shower probability by time of day
            #
            # Morning:  00:00-10:59
            # Daytime:  11:00-17:59
            # Evening:  18:00-23:59
            # ---------------------------------------------------------
            if hour < 11:

                period = "morning"
                p_use = (
                    morning_shower_use_probability
                )

            elif hour < 18:

                period = "daytime"
                p_use = (
                    daytime_shower_use_probability
                )

            else:

                period = "evening"
                p_use = (
                    evening_shower_use_probability
                )



            # -----------------------------------------
            # Daily maximum: one shower/person/day.
            # -----------------------------------------
            blocked_by_daily_limit = (
                day in shower_dates
            )

            shower_assigned = False
            shower_energy_kwh_th = 0.0

            if not blocked_by_daily_limit:

                if random.random() < p_use:

                    shower_energy_kwh_th = random.uniform(
                        min_consumption / eta_boiler,
                        max_consumption / eta_boiler,
                    )

                    # Energy of ONE whole shower is assigned once,
                    # at the first timestep of the Personal-care event.
                    dhw_df.iat[
                        start_pos,
                        dhw_df.columns.get_loc(person),
                    ] = shower_energy_kwh_th

                    shower_dates.add(day)

                    shower_assigned = True

            household_type = (
                household_type_by_household.get(
                    household_id,
                    "Unknown",
                )
                if household_type_by_household is not None
                else "Unknown"
            )

            audit_rows.append({
                "event_id": event_id,

                "person": person,
                "household": household_id,

                "household_type": household_type,
                "family_size": family_size_by_household.get(
                    household_id,
                    np.nan,
                ),

                "date": day,

                "personal_care_event_start": start_ts,
                "personal_care_event_end": end_ts,

                "personal_care_event_start_ts": start_pos,
                "personal_care_event_end_ts": end_pos,

                # Number of actual activity-code-2 occurrences
                # merged into this event.
                "n_personal_care_occurrences_in_event": len(cluster),

                # Span from first to last PC occurrence.
                "personal_care_event_span_ts": (
                    end_pos - start_pos + 1
                ),

                # Number of distinct Personal-care events of this
                # person on this calendar day.
                "n_personal_care_events_same_person_day": (
                    n_events_same_day
                ),

                "period": period,
                "shower_probability": p_use,

                "blocked_by_daily_shower_limit": (
                    blocked_by_daily_limit
                ),

                "raw_shower_generated": shower_assigned,

                "raw_shower_energy_kwh_th": (
                    shower_energy_kwh_th
                ),
            })

    event_audit_df = pd.DataFrame(
        audit_rows
    )

    # -------------------------------------------------------------
    # How many RAW showers did the same person have on the same day?
    #
    # With the new daily constraint this should never exceed 1.
    # Keeping the column is a useful consistency check.
    # -------------------------------------------------------------
    if not event_audit_df.empty:

        raw_only = event_audit_df[
            event_audit_df["raw_shower_generated"]
        ].copy()

        if not raw_only.empty:

            raw_counts = (
                raw_only
                .groupby(
                    ["person", "date"]
                )
                .size()
                .rename(
                    "n_raw_showers_same_person_day"
                )
            )

            event_audit_df = (
                event_audit_df
                .merge(
                    raw_counts,
                    how="left",
                    left_on=["person", "date"],
                    right_index=True,
                )
            )

        else:
            event_audit_df[
                "n_raw_showers_same_person_day"
            ] = 0

        event_audit_df[
            "n_raw_showers_same_person_day"
        ] = (
            event_audit_df[
                "n_raw_showers_same_person_day"
            ]
            .fillna(0)
            .astype(int)
        )

    return (
        dhw_df,
        event_audit_df,
    )


# Function to build DHW boiler specifications for each household.
def build_dhw_boiler_specs_by_household(
    household_type_by_household: Dict[str, str],
    dhw_specs: Dict[str, Tuple]
) -> Dict[str, Dict[str, float]]:
    """
    Build the mapping Household_X -> DHW boiler specifications from the
    household type assigned to each household.

    Compatible with both 3-value dhw_specs entries:
        (equivalent_people, capacity_kwh_th, recharge_power_kw)

    and 4-value dhw_specs entries:
        (equivalent_people, capacity_kwh_th, recharge_power_kw, loss_rate_h)

    In Module 03, loss_rate_h is not used directly to build DHW demand.
    It is kept for downstream use in Module 04 / HAGRID.
    """
    boiler_specs = {}

    for household_id, household_type in household_type_by_household.items():
        if household_type not in dhw_specs:
            raise KeyError(f"Household type '{household_type}' not found in dhw_specs.")

        values = dhw_specs[household_type]

        if len(values) == 3:
            equivalent_people, capacity_kwh_th, recharge_power_kw = values
            loss_rate_h = None

        elif len(values) >= 4:
            equivalent_people, capacity_kwh_th, recharge_power_kw, loss_rate_h = values[:4]

        else:
            raise ValueError(
                f"dhw_specs['{household_type}'] must contain at least 3 values: "
                "(equivalent_people, capacity_kwh_th, recharge_power_kw). "
                f"Found: {values}"
            )

        boiler_specs[household_id] = {
            "household_type": household_type,
            "equivalent_people": int(equivalent_people),
            "capacity_kwh_th": float(capacity_kwh_th),
            "recharge_power_kw": float(recharge_power_kw),
        }

        if loss_rate_h is not None:
            boiler_specs[household_id]["loss_rate_h"] = float(loss_rate_h)

    return boiler_specs


# Function to shift simultaneous or time-close showers when
# boiler capacity is not plausible.
def resolve_dhw_overlap_by_household(
    person_dhw_df: pd.DataFrame,
    dhw_boiler_specs_by_household: Dict[str, Dict[str, float]],
    simultaneous_probability: float,
    step_minutes: int = 10,
    alpha_util_boiler: float = 0.75,
    reference_shower_energy_kwh: float = None,
    use_average_shower_for_shift: bool = False,
    gap_reset_cluster_ts: int = 12,
    max_shift_steps: int = 72,
    generation_audit_df: pd.DataFrame = None,
):
    """
    Post-process RAW DHW shower demand to make shower timing plausible.

    Returns
    -------
    df_out
        Final person-level DHW demand after shifting.

    shift_audit_df
        One row per RAW shower, with original/final timestep and
        reason for any shift.
    """

    if person_dhw_df.empty:
        return (
            person_dhw_df.copy(),
            pd.DataFrame(),
        )

    df_out = pd.DataFrame(
        0.0,
        index=person_dhw_df.index,
        columns=person_dhw_df.columns,
        dtype=float,
    )

    step_h = step_minutes / 60.0
    n_steps = len(person_dhw_df.index)

    household_of_person = (
        person_dhw_df.columns
        .to_series()
        .str.extract(
            r"^(Household_\d+)"
        )[0]
    )

    households = (
        household_of_person
        .dropna()
        .unique()
    )

    # -------------------------------------------------------------
    # Lookup RAW event IDs generated by the previous function.
    # Key = (person, raw timestep index)
    # -------------------------------------------------------------
    raw_event_lookup = {}

    raw_generation_lookup = {}

    if (
        generation_audit_df is not None
        and not generation_audit_df.empty
    ):

        raw_generation = (
            generation_audit_df[
                generation_audit_df[
                    "raw_shower_generated"
                ]
            ]
            .copy()
        )

        for _, row in raw_generation.iterrows():

            key = (
                row["person"],
                int(
                    row[
                        "personal_care_event_start_ts"
                    ]
                ),
            )

            raw_event_lookup[key] = (
                row["event_id"]
            )

            raw_generation_lookup[
                row["event_id"]
            ] = row.to_dict()

    # -------------------------------------------------------------
    # Persistent state for every shower while it is possibly shifted
    # several times.
    # -------------------------------------------------------------
    event_state = {}

    final_audit_rows = []

    fallback_event_counter = 0

    for household_id in households:

        cols = (
            household_of_person[
                household_of_person
                == household_id
            ]
            .index
            .tolist()
        )

        if len(cols) == 0:
            continue

        if household_id not in dhw_boiler_specs_by_household:
            raise KeyError(
                "Missing DHW boiler specifications for "
                f"household '{household_id}'."
            )

        specs = (
            dhw_boiler_specs_by_household[
                household_id
            ]
        )

        household_type = specs.get(
            "household_type",
            "Unknown",
        )

        # REAL number of members.
        family_size = len(cols)

        boiler_capacity_kwh_th = float(
            specs["capacity_kwh_th"]
        )

        recharge_power_kw = float(
            specs["recharge_power_kw"]
        )

        usable_cluster_budget_kwh = (
            alpha_util_boiler
            * boiler_capacity_kwh_th
        )

        charging_energy_per_step = (
            recharge_power_kw
            * step_h
        )

        # bucket[t] contains:
        #
        # (
        #   event_id,
        #   person,
        #   shower_energy,
        #   original_timestep
        # )
        bucket = {}

        for person in cols:

            vals = (
                person_dhw_df[person]
                .to_numpy(dtype=float)
            )

            nonzero_indices = (
                np.flatnonzero(vals > 0)
            )

            for t_idx_raw in nonzero_indices:

                t_idx = int(t_idx_raw)

                event_id = raw_event_lookup.get(
                    (person, t_idx)
                )

                if event_id is None:

                    fallback_event_counter += 1

                    event_id = (
                        "DHW_FALLBACK_"
                        f"{fallback_event_counter:06d}"
                    )

                event_state[event_id] = {
                    "shift_reasons": set(),
                    "n_shift_operations": 0,
                    "n_placement_attempts": 0,
                }

                bucket.setdefault(
                    t_idx,
                    [],
                ).append(
                    (
                        event_id,
                        person,
                        float(vals[t_idx]),
                        t_idx,
                    )
                )

        # ---------------------------------------------------------
        # Existing cluster logic
        # ---------------------------------------------------------
        cluster_energy_debt_kwh = 0.0
        last_accepted_ts = None

        for t_idx in range(n_steps):

            if t_idx not in bucket:
                continue

            queue = bucket.pop(t_idx)

            if last_accepted_ts is None:

                cluster_energy_debt_kwh = 0.0

            else:

                gap_ts = (
                    t_idx
                    - last_accepted_ts
                )

                if gap_ts >= gap_reset_cluster_ts:

                    cluster_energy_debt_kwh = 0.0

                else:

                    cluster_energy_debt_kwh = max(
                        0.0,
                        cluster_energy_debt_kwh
                        - gap_ts
                        * charging_energy_per_step,
                    )

            accepted_showers_in_ts = 0

            while queue:

                (
                    event_id,
                    person,
                    shower_energy,
                    original_timestep,
                ) = queue.pop(0)

                state = event_state[event_id]

                state[
                    "n_placement_attempts"
                ] += 1

                # -------------------------------------------------
                # Reason 1: simultaneous shower
                # -------------------------------------------------
                simultaneity_not_allowed = (
                    accepted_showers_in_ts >= 1
                    and random.random()
                    >= simultaneous_probability
                )

                if (
                    use_average_shower_for_shift
                    and reference_shower_energy_kwh
                    is not None
                ):
                    shift_energy = (
                        reference_shower_energy_kwh
                    )
                else:
                    shift_energy = shower_energy

                # -------------------------------------------------
                # Reason 2: boiler energy budget
                # -------------------------------------------------
                exceeds_cluster_budget = (
                    cluster_energy_debt_kwh
                    + shower_energy
                ) > (
                    usable_cluster_budget_kwh
                    + 1e-12
                )

                # -------------------------------------------------
                # ACCEPT CURRENT POSITION
                # -------------------------------------------------
                if (
                    not simultaneity_not_allowed
                    and not exceeds_cluster_budget
                ):

                    df_out.iat[
                        t_idx,
                        df_out.columns.get_loc(
                            person
                        ),
                    ] += shower_energy

                    cluster_energy_debt_kwh += (
                        shower_energy
                    )

                    last_accepted_ts = t_idx

                    accepted_showers_in_ts += 1

                    reasons = (
                        state["shift_reasons"]
                    )

                    if len(reasons) == 0:
                        reason_label = "none"

                    elif reasons == {"simultaneity"}:
                        reason_label = "simultaneity"

                    elif reasons == {"energy_budget"}:
                        reason_label = "energy_budget"

                    else:
                        reason_label = "both"

                    raw_info = (
                        raw_generation_lookup.get(
                            event_id,
                            {},
                        )
                    )

                    final_audit_rows.append({
                        "event_id": event_id,

                        "person": person,
                        "household": household_id,

                        "household_type": (
                            household_type
                        ),
                        "family_size": (
                            family_size
                        ),

                        "shower_energy_kwh_th": (
                            shower_energy
                        ),

                        "raw_timestep": (
                            original_timestep
                        ),
                        "final_timestep": (
                            t_idx
                        ),

                        "raw_datetime": (
                            person_dhw_df.index[
                                original_timestep
                            ]
                        ),
                        "final_datetime": (
                            person_dhw_df.index[
                                t_idx
                            ]
                        ),

                        "shifted": (
                            t_idx
                            != original_timestep
                        ),

                        "shift_steps_total": (
                            t_idx
                            - original_timestep
                        ),

                        "shift_minutes_total": (
                            (
                                t_idx
                                - original_timestep
                            )
                            * step_minutes
                        ),

                        "n_shift_operations": (
                            state[
                                "n_shift_operations"
                            ]
                        ),

                        "n_placement_attempts": (
                            state[
                                "n_placement_attempts"
                            ]
                        ),

                        "shift_reason": (
                            reason_label
                        ),

                        "shift_due_to_simultaneity": (
                            "simultaneity"
                            in reasons
                        ),

                        "shift_due_to_energy_budget": (
                            "energy_budget"
                            in reasons
                        ),

                        # Information inherited from RAW-generation
                        # audit.
                        "n_personal_care_events_same_person_day": (
                            raw_info.get(
                                "n_personal_care_events_same_person_day",
                                np.nan,
                            )
                        ),

                        "n_personal_care_occurrences_in_event": (
                            raw_info.get(
                                "n_personal_care_occurrences_in_event",
                                np.nan,
                            )
                        ),

                        "n_raw_showers_same_person_day": (
                            raw_info.get(
                                "n_raw_showers_same_person_day",
                                np.nan,
                            )
                        ),

                        "period": (
                            raw_info.get(
                                "period",
                                None,
                            )
                        ),

                        "shower_probability": (
                            raw_info.get(
                                "shower_probability",
                                np.nan,
                            )
                        ),
                    })

                    continue

                # -------------------------------------------------
                # SHIFT REQUIRED
                # -------------------------------------------------
                current_reasons = set()

                shift_steps = (
                    1
                    if simultaneity_not_allowed
                    else 0
                )

                if simultaneity_not_allowed:
                    current_reasons.add(
                        "simultaneity"
                    )

                if exceeds_cluster_budget:

                    current_reasons.add(
                        "energy_budget"
                    )

                    excess_kwh = max(
                        0.0,
                        (
                            cluster_energy_debt_kwh
                            + shower_energy
                            - usable_cluster_budget_kwh
                        ),
                    )

                    if charging_energy_per_step > 0:

                        shift_for_budget = int(
                            np.ceil(
                                excess_kwh
                                / charging_energy_per_step
                            )
                        )

                        shift_for_shower_recharge = int(
                            np.ceil(
                                shift_energy
                                / charging_energy_per_step
                            )
                        )

                    else:

                        shift_for_budget = 1
                        shift_for_shower_recharge = 1

                    shift_steps = max(
                        shift_steps,
                        shift_for_budget,
                        shift_for_shower_recharge,
                    )

                shift_steps = max(
                    1,
                    shift_steps,
                )

                if max_shift_steps is not None:

                    shift_steps = min(
                        shift_steps,
                        max_shift_steps,
                    )

                state[
                    "shift_reasons"
                ].update(
                    current_reasons
                )

                state[
                    "n_shift_operations"
                ] += 1

                new_idx = (
                    t_idx
                    + shift_steps
                )

                if new_idx >= n_steps:
                    new_idx = (
                        n_steps - 1
                    )

                bucket.setdefault(
                    new_idx,
                    [],
                ).append(
                    (
                        event_id,
                        person,
                        shower_energy,
                        original_timestep,
                    )
                )

    shift_audit_df = pd.DataFrame(
        final_audit_rows
    )

    return (
        df_out,
        shift_audit_df,
    )

# --------
#  EV

# Function to calculate EV home-charging electricity consumption.
def calculate_ev_consumption_by_season(
    activity_df: pd.DataFrame,
    df_occ: pd.DataFrame,
    ev_presence_by_household: dict,
    car_use_probability: float,
    km_per_step: float,
    # Consumption._per_km: float,
    consumption_by_household: dict,
    # battery_capacity_kwh: float,
    battery_capacity_by_household: dict,
    workplace_charging_probability: float,
    home_charging_power_kw: float,
    workplace_charging_power_kw: float,
    # initial_soc: float,
    charging_efficiency: float = 0.99,
    min_dwell_steps: int = 6,

    lookahead: int = 6
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
    - df_charge_home_greedy: kWh of home charging 
        (this already represents the EV battery charging profile 
        accounting for occupancy -> maximum charging
        as soon as charging is possible)
    - df_charge_work: kWh of workplace charging
    - df_soc: battery state of charge [%] at each timestep
    """
    # Hours per timestep.
    step_h = 10/60
    # Energy charged at home and at work.
    home_charging_per_step   = home_charging_power_kw   * step_h * charging_efficiency
    workplace_charging_per_step = workplace_charging_power_kw * step_h * charging_efficiency
    
    # Initialize DataFrames.
    df_charge_home_greedy = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
    df_charge_work = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
    df_soc         = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
    df_km         = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
    ev_demand_df = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)




    n_ts = len(activity_df)
    # ===== Person-level pre-assignments, performed once =====
    initial_soc_by_person = {}
    energy_to_full_by_person = {}

    # Build metadata rows using the activity_df column names, e.g. 'Household_1_Member_1_...').
    ev_metadata_rows = []

    for person in activity_df.columns:
        # Consider only Member_1, consistently with the rest of the code. Remove this filter to include all members.
        if not re.match(r'Household_\d+_Member_1_', person):
            continue

        simple_household_id = re.match(r'(Household_\d+)', person).group(1)
        has_ev = ev_presence_by_household.get(simple_household_id, False)
        cap = battery_capacity_by_household.get(simple_household_id, None)
        km_consumption = consumption_by_household.get(simple_household_id, None)
        if has_ev and (cap is not None):
            # soc0 = float(random.uniform(0.95,1))
            soc0 = 1 # Assume all cars are initially fully charged, otherwise a simultaneous peak appears at the beginning of the week.
            initial_soc_by_person[person] = soc0
            energy_to_full_by_person[person] = (1.0 - soc0) * cap

        else:
            initial_soc_by_person[person] = np.nan
            energy_to_full_by_person[person] = 0.0

        ev_metadata_rows.append({
            "person": person,
            "household": simple_household_id,
            "has_ev": bool(has_ev),
            "bev_capacity_kWh": cap,
            "cons_kWh_per_km": km_consumption,
            "soc_initial": initial_soc_by_person.get(person, np.nan),
            "energy_to_full_kWh": energy_to_full_by_person.get(person, 0.0)

        })

    for person in activity_df.columns:
        # Only Member_1.
        if not re.match(r'Household_\d+_Member_1_', person):
            continue

        # Check whether the household owns an EV.
        simple_household_id = re.match(r'(Household_\d+)', person).group(1)
        if not ev_presence_by_household.get(simple_household_id, False):
            continue

        # Retrieve household-specific EV parameters.
        battery_capacity_kwh = battery_capacity_by_household[simple_household_id]
        consumption_per_km   = consumption_by_household[simple_household_id]    

        # Initial state of charge. 
        # initial_soc = random.uniform(0.95,1) # random, otherwise a large demand peak appears for all EVs at the start of the week
        # soc = initial_soc * battery_capacity_kwh  # Initial SoC in kWh
        soc = initial_soc_by_person[person] * battery_capacity_kwh
        dwell = 0                           # counter of consecutive timesteps at home
        work_block = False                  # whether the EV is currently in a workplace-charging block


        vals     = activity_df[person].values
        occ_vals = df_occ[person].values

        for t in range(n_ts):
            # First, record the SoC as a percentage.
            df_soc.iat[t, df_soc.columns.get_loc(person)] = soc/battery_capacity_kwh*100

            code = vals[t]
            
            # 1) If travelling -> driving consumption.
            if code == 7:
                dwell = 0
                work_block = False
                if random.random() < car_use_probability:
                    # Calculate electricity consumed while travelling.
                    e_drive = km_per_step * consumption_per_km
                    ev_demand_df.iat[t, ev_demand_df.columns.get_loc(person)] = e_drive                      
                    
                    # Consumption.
                    soc = max(0.0, soc - km_per_step*consumption_per_km)
                    # Distance travelled in each timestep.
                    df_km.iat[t, df_km.columns.get_loc(person)] = km_per_step        
                
                continue

            # 2) If the activity is Work and there is no travel in the next lookahead timesteps.
            if code == 14:
                dwell = 0
                # Check that there is no immediate travel.
                next_slice = vals[t+1:t+1+lookahead]
                if 7 not in next_slice and random.random()<workplace_charging_probability:
                    work_block = True
                else:
                    work_block = False

            # 3) If the EV is in a workplace-charging block.
            if work_block:
                if soc < battery_capacity_kwh:
                    df_charge_work.iat[t, df_charge_work.columns.get_loc(person)] = workplace_charging_per_step
                    soc = min(battery_capacity_kwh, soc + workplace_charging_per_step)
                # Exit the workplace-charging block as soon as the activity changes.
                if code != 14:
                    work_block = False
                continue

            # 4) Otherwise, count consecutive timesteps at home.
            if occ_vals[t]:
                dwell += 1
            else:
                dwell = 0


            # If the minimum dwell time at home is reached and the battery is not full, charge at home.
            if dwell >= min_dwell_steps and soc < battery_capacity_kwh:
                df_charge_home_greedy.iat[t, df_charge_home_greedy.columns.get_loc(person)] = home_charging_per_step
                soc = min(battery_capacity_kwh, soc + home_charging_per_step)

        # end loop t
        # # Complete df_soc with the last value for possible NaN values.
        # df_soc[person].ffill(inplace=True)
    # More efficient alternative without FutureWarning.
    # Fill all df_soc NaN values with forward-fill in one vectorized operation.
    df_soc = df_soc.ffill()


    # ===== EV diagnostics and metadata export =====
    ev_metadata_df = pd.DataFrame(ev_metadata_rows).set_index("person")
    # Save EV metadata using the globally defined output folder.
    try:
        out_meta = os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_ev_metadata_by_person.xlsx')
        ev_metadata_df.to_excel(out_meta)
    except Exception:
        # Do not stop the simulation for a metadata-writing error.
        print("[WARN] Unable to save EV metadata. Check additional_consumption_output_xls_folder.")

    return df_charge_home_greedy, df_charge_work, df_soc, df_km, ev_demand_df




# --------
#  HEATING

# Function to calculate annual thermal demand for heating for each household, once.
def calculate_annual_thermal_demand_by_household(
    household_type_by_household: Dict[str,str],
    household_apartment_specs: Dict[str, Tuple[int,float,float]],
    min_thermal_demand_per_m2: float,
    max_thermal_demand_per_m2: float,
    seed: int = None
) -> Dict[str, float]:
    """
    Calculate and return a dictionary { 'Household_X': annual_thermal_demand_kWh_th }
    using the same logic previously used inside calculate_heating_consumption_by_season.
    If a seed is provided, set np.random.seed(seed) to make random generation reproducible.
    """
    if seed is not None:
        np.random.seed(seed)

    annual_thermal_demand = {}
    for household_id, ftype in household_type_by_household.items():
        if ftype not in household_apartment_specs:
            raise KeyError(f"Household type '{ftype}' not found")
        _, a_min, a_max = household_apartment_specs[ftype]
        area = np.random.uniform(a_min, a_max)
        consumption_per_m2 = np.random.uniform(min_thermal_demand_per_m2, max_thermal_demand_per_m2)
        annual_thermal_demand[household_id] = area * consumption_per_m2  # kWh_th/year

    return annual_thermal_demand



# Function to calculate heat-pump electricity consumption for heating.
def calculate_heating_consumption_by_season(
    thermal_profile_cop_excel_path: str,
    annual_thermal_demand: Dict[str, float],
    season: str,
    season_periods: Dict[str, Tuple[str,str]],
    df_occ: pd.DataFrame,               # 10-minute occupancy for all members
    min_absent_steps: int               # minimum consecutive timesteps with nobody at home

) -> pd.DataFrame:
    """
    Return a 10-minute DataFrame with heating electricity demand, kWh_el, for each household and season.
    It now takes annual_thermal_demand as input, calculated once outside the function.
    - thermal_profile_cop_excel_path: Excel path with sheet "thermal_profile" or legacy "profilo_termico", with columns:
        * 'fabb_norm' (8760h, sum=1)
        * 'COP'       (hourly)
    - annual_thermal_demand: dict { 'Household_X': kWh_th/year } PRE-COMPILED
    - season: season name (key in season_periods e cop_pdc)
    - season_periods: season -> (start_datetime, end_datetime)
    - df_occ: member presence at 10-minute resolution
    """
    # 1) Read the normalized hourly thermal profile and COP.
    # Read the normalized hourly thermal profile and COP.
    # The English sheet name is preferred, while the legacy Italian sheet name is
    # retained as a fallback for compatibility with older input files.
    try:
        df_prof = pd.read_excel(
            thermal_profile_cop_excel_path,
            sheet_name="thermal_profile",
            index_col=0,
            parse_dates=True
        )
    except ValueError:
        df_prof = pd.read_excel(
            thermal_profile_cop_excel_path,
            sheet_name="profilo_termico",
            index_col=0,
            parse_dates=True
        )

    df_prof = df_prof.loc[:, ['fabb_norm', 'COP']]

    # 2) Restrict to the selected seasonal period.
    start, end = season_periods[season]
    df_prof = df_prof.loc[start:end]

    # 3) Distribute annual demand over hourly timesteps.
    # DataFrame with hourly resolution and N households.
    df_annual_th = pd.DataFrame(
        { household_id: df_prof['fabb_norm'] * annual_estimate
        for household_id, annual_estimate in annual_thermal_demand.items() },
        index=df_prof.index
    )

    # 4) Create a 10-minute index and split each hour into six 10-minute steps.
    idx_10min = df_occ.index
    df_annual_10min = (
        df_annual_th
        .reindex(idx_10min, method='ffill')   # spreading
        / 6                                   # kWh_th per 10-minute timestep
    )

    # 5) Convert hourly COP to 10-minute resolution with forward-fill.
    cop_10min = df_prof['COP'].reindex(idx_10min).ffill().bfill()

    # 6) Convert to electricity: kWh_el = kWh_th / COP.
    df_heating_el_10min = df_annual_10min.div(cop_10min, axis=0)    
    
    # 7) For each household, derive at-home presence: 
    #    from df_occ, with columns for each member, create a binary household series equal to 1 if at least one member is at home.
    household_presence = {}
    for household_id in household_type_by_household:
        # Select all members of that household.
        cols = [c for c in df_occ.columns if c.startswith(household_id + "_")]
        # For each timestep, 1 if at least one member is at home.
        household_presence[household_id] = df_occ[cols].max(axis=1)
    household_presence_df = pd.DataFrame(household_presence, index=df_occ.index)

    # 8) Identify runs where all members are absent for each household.
    absence_df = 1 - household_presence_df  # 1 = all absent, 0 = at least one member at home
    df_heat_el_masked = df_heating_el_10min.copy()

    for household_id in absence_df.columns:
        s = absence_df[household_id]
        # Label constant True/False blocks.
        block_id = (s != s.shift(fill_value=0)).cumsum()
        # For each block, calculate its length and whether it is an absence block.
        lengths = s.groupby(block_id).transform('sum')
        # Here lengths[i] is the number of consecutive timesteps equal to s[i], counting only True values.
        # Select all points where s==1 and the block length is >= min_absent_steps.
        to_zero = (s == 1) & (lengths >= min_absent_steps)
        df_heat_el_masked.loc[to_zero, household_id] = 0.0  
        
    return df_heat_el_masked
#---------------------------------------------------------------------------------------------------------------------------------------------




#-------------------------------------------------------------------
#       Sec. 3.6 -- 4.3) FUNCTIONS FOR AGGREGATING ADDITIONAL LOADS
#-------------------------------------------------------------------

def persons_df_to_household_df(
    df_person: pd.DataFrame,
    household_index: pd.Index
) -> pd.DataFrame:
    """
    Convert a person-level DataFrame into a household-level DataFrame.

    Parameters
    ----------
    df_person : pd.DataFrame
        DataFrame with one column per person, e.g.
        "Household_1_Member_1_...".
    household_index : pd.Index
        Ordered household index to use in the output columns.

    Returns
    -------
    pd.DataFrame
        DataFrame with the same index as df_person and one column per household.
        Missing households are filled with zeros.
    """
    if df_person.shape[1] == 0:
        return pd.DataFrame(
            0.0,
            index=df_person.index,
            columns=household_index,
            dtype=float
        )

    cols = df_person.columns.to_series()
    household_series = cols.str.extract(r'^(Household_\d+)')[0]

    valid_mask = household_series.notna()

    if not valid_mask.any():
        return pd.DataFrame(
            0.0,
            index=df_person.index,
            columns=household_index,
            dtype=float
        )

    df_valid = df_person.loc[:, valid_mask.index[valid_mask]]
    household_for_valid_columns = household_series.loc[df_valid.columns]

    # Build a binary person-to-household matrix and aggregate via matrix product.
    aggregation_matrix = (
        pd.get_dummies(household_for_valid_columns)
        .reindex(columns=sorted(household_for_valid_columns.unique()))
        .astype(float)
    )

    arr = df_valid.values.dot(aggregation_matrix.values)

    household_df = pd.DataFrame(
        arr,
        index=df_valid.index,
        columns=aggregation_matrix.columns,
        dtype=float
    )

    household_df = household_df.reindex(columns=household_index, fill_value=0.0)

    return household_df


#---------------------------------------------------------------------------------------------------------------------------------------------

### 3.6.5 F(x) Optional heating-profile preprocessing

In [ ]:
# (optional) Function to build and save the normalized annual thermal-demand profile from climate data.
def build_normalized_thermal_profile(
        heating_coefficients_path,
        temperature_data_path,
        thermal_profile_cop_excel_path):

    coeff_df = pd.read_excel(
        heating_coefficients_path,
        index_col=0,
    )

    temperature_df = (
        pd.read_excel(
            temperature_data_path,
            index_col=0,
            parse_dates=True,
        )
        .rename(
            columns={
                "Temperatura [°C]": "T",
                "COP": "COP",
            }
        )
    )

    required_temperature_columns = {
        "T",
        "COP",
    }

    missing_temperature_columns = (
        required_temperature_columns
        - set(temperature_df.columns)
    )

    if missing_temperature_columns:
        raise KeyError(
            "Missing climate-data columns: "
            f"{sorted(missing_temperature_columns)}"
        )

    required_coefficient_rows = {
        "Angular coeff",
        "Intercept",
    }

    missing_coefficient_rows = (
        required_coefficient_rows
        - set(coeff_df.index)
    )

    if missing_coefficient_rows:
        raise KeyError(
            "Missing heating-coefficient rows: "
            f"{sorted(missing_coefficient_rows)}"
        )

    temperature_df = temperature_df.loc[
        :,
        ["T", "COP"],
    ].copy()

    datetime_index = temperature_df.index
    hourly_positions = datetime_index.hour.to_numpy()

    angular_coefficients = (
        coeff_df.loc["Angular coeff"]
        .to_numpy(dtype=float)
    )

    intercepts = (
        coeff_df.loc["Intercept"]
        .to_numpy(dtype=float)
    )

    if (
        len(angular_coefficients) != 24
        or len(intercepts) != 24
    ):
        raise ValueError(
            "Heating coefficients must contain exactly "
            "24 hourly values."
        )

    months = datetime_index.month
    days = datetime_index.day

    heating_period_mask = (
        ((months == 11) & (days >= 15))
        | (months == 12)
        | (months <= 3)
        | ((months == 4) & (days <= 15))
    )

    temperature_mask = (
        temperature_df["T"].to_numpy(dtype=float)
        <= 16.0
    )

    unit_thermal_demand = (
        angular_coefficients[hourly_positions]
        * temperature_df["T"].to_numpy(dtype=float)
        + intercepts[hourly_positions]
    )

    valid_heating_mask = (
        heating_period_mask
        & temperature_mask
    )

    unit_thermal_demand[
        ~valid_heating_mask
    ] = 0.0

    unit_thermal_demand = np.clip(
        unit_thermal_demand,
        0.0,
        None,
    )

    total_unit_demand = (
        unit_thermal_demand.sum()
    )

    if total_unit_demand <= 0:
        raise ValueError(
            "The reconstructed thermal-demand profile "
            "has a non-positive annual sum."
        )

    normalized_thermal_demand = (
        unit_thermal_demand
        / total_unit_demand
    )

    thermal_profile_df = pd.DataFrame(
        {
            "fabb_unitario_th": unit_thermal_demand,
            "fabb_norm": normalized_thermal_demand,
            "COP": temperature_df["COP"].to_numpy(
                dtype=float
            ),
        },
        index=datetime_index,
    )

    output_parent = os.path.dirname(
        thermal_profile_cop_excel_path
    )

    if output_parent:
        os.makedirs(
            output_parent,
            exist_ok=True,
        )

    thermal_profile_df.to_excel(
        thermal_profile_cop_excel_path,
        sheet_name="thermal_profile",
    )

    return thermal_profile_df

## 3.7 Additional-load calculation

This section calculates and saves occupancy, DHW, cooking, EV, cooling and
heating profiles, together with weekly aggregated profiles and annual summaries.
No plots are generated in this section.

In [ ]:
# -----------------------------------------------------------------------------
#   Sec 3.7 -- 0) Optional preprocessing: normalized heating profile
# -----------------------------------------------------------------------------
if RUN_REBUILD_THERMAL_PROFILE:

    required_heating_input_files = (
        heating_coefficients_path,
        temperature_data_path,
    )

    missing_heating_input_files = [
        path
        for path in required_heating_input_files
        if not os.path.isfile(path)
    ]

    if missing_heating_input_files:
        raise FileNotFoundError(
            "Heating-profile inputs are missing:\n - "
            + "\n - ".join(missing_heating_input_files)
        )

    rebuilt_thermal_profile_df = build_normalized_thermal_profile(
        heating_coefficients_path=heating_coefficients_path,
        temperature_data_path=temperature_data_path,
        thermal_profile_cop_excel_path=thermal_profile_cop_excel_path,
    )

    print(
        "Normalized heating profile rebuilt and saved to:\n"
        f"{thermal_profile_cop_excel_path}\n"
        "Annual normalized sum: "
        f"{rebuilt_thermal_profile_df['fabb_norm'].sum():.6f}"
    )


# =============================================================================
#   Sec 3.7 -- 1) Additional-load calculation
# =============================================================================
if RUN_ADDITIONAL_LOADS:

    # -------------------------------------------------------------------------
    #   Sec 3.7 -- 1.1) Validate and load input files
    # -------------------------------------------------------------------------
    required_input_files = (
        household_profiles_file,
        cooling_profiles_file,
        thermal_profile_cop_excel_path,
    )

    missing_input_files = [
        path
        for path in required_input_files
        if not os.path.isfile(path)
    ]

    if missing_input_files:
        raise FileNotFoundError(
            "Required MODULE_03 input files are missing:\n - "
            + "\n - ".join(missing_input_files)
        )

    cooling_week_df = pd.read_excel(
        cooling_profiles_file,
        sheet_name="EE cool cons kWh_el per capita",
        index_col="h",
    )

    # Read all household-activity sheets once and reuse them for every season.
    sheets_dict = pd.read_excel(
        household_profiles_file,
        sheet_name=None,
    )


    # -------------------------------------------------------------------------
    #   Sec 3.7 -- 1.2) Identify households and assign technologies
    # -------------------------------------------------------------------------
    # Household composition and technology assignments are defined once and
    # then kept unchanged across the four representative seasons.
    first_season = selectable_periods[0]

    first_activity_sheet = next(
        (
            sheet_name
            for sheet_name in sheets_dict
            if canonical_season(sheet_name) == first_season
        ),
        None,
    )

    if first_activity_sheet is None:
        raise KeyError(
            f"No activity-profile sheet found for season '{first_season}'. "
            f"Available sheets: {list(sheets_dict)}"
        )

    initial_activity_df = sheets_dict[first_activity_sheet].copy()
    initial_activity_df["Datetime"] = pd.to_datetime(
        initial_activity_df["Datetime"]
    )
    initial_activity_df.set_index(
        "Datetime",
        inplace=True,
    )
    initial_activity_df = normalize_household_profile_columns(
        initial_activity_df,
        default_season=first_season,
    )

    # Map each person column to its simple household identifier and type.
    households0, household_types0 = identify_households(
        initial_activity_df
    )

    household_type_by_household = {
        household_id: canonical_household_type(
            household_types0[person_column]
        )
        for person_column, household_id in households0.items()
    }

    # Base person identifiers, e.g. Household_1_Member_1.
    base_person_ids = (
        initial_activity_df.columns
        .to_series()
        .str.extract(r"^(Household_\d+_Member_\d+)_")[0]
        .dropna()
        .unique()
        .tolist()
    )

    simple_households = sorted(
        households0
        .dropna()
        .astype(str)
        .unique()
        .tolist(),
        key=lambda household_id: int(
            household_id.split("_")[1]
        ),
    )

    # Household-specific ESWH parameters.
    dhw_boiler_specs_by_household = (
        build_dhw_boiler_specs_by_household(
            household_type_by_household,
            dhw_specs,
        )
    )

    # Technology ownership is sampled once and retained across all seasons.
    shower_presence_by_household = {
        household_id: (
            random.random() < shower_ownership_probability
        )
        for household_id in simple_households
    }

    induction_presence_by_household = {
        household_id: (
            random.random() < induction_ownership_probability
        )
        for household_id in simple_households
    }

    ev_presence_by_household = {
        household_id: (
            random.random() < ev_ownership_probability
        )
        for household_id in simple_households
    }

    # Assign one EV class to every EV-equipped household.
    ev_class_by_household = {}
    battery_capacity_by_household = {}
    consumption_by_household = {}

    for household_id in simple_households:
        if ev_presence_by_household[household_id]:
            household_type = household_type_by_household[household_id]

            possible_ev_classes = (
                ev_classes_by_household_type.get(
                    household_type,
                    list(ev_classes),
                )
            )

            selected_ev_class = random.choice(
                possible_ev_classes
            )

            ev_class_by_household[household_id] = (
                selected_ev_class
            )
            battery_capacity_by_household[household_id] = (
                ev_classes[selected_ev_class]["capacity"]
            )
            consumption_by_household[household_id] = (
                ev_classes[selected_ev_class]["cons_kwh_per_km"]
            )

        else:
            ev_class_by_household[household_id] = None
            battery_capacity_by_household[household_id] = None
            consumption_by_household[household_id] = None

    # Accumulate distance travelled across the four representative weeks.
    weekly_km = pd.Series(
        0.0,
        index=base_person_ids,
        dtype=float,
    )

    # Calculate the annual thermal demand once and use exactly the same values
    # for all seasonal heating profiles and for the annual summary.
    seed_for_heating = np.random.randint(
        0,
        2**31 - 1,
    )

    annual_thermal_demand_th = (
        calculate_annual_thermal_demand_by_household(
            household_type_by_household,
            household_apartment_specs,
            min_thermal_demand_per_m2,
            max_thermal_demand_per_m2,
            seed=seed_for_heating,
        )
    )

    household_index = pd.Index(
        sorted(
            annual_thermal_demand_th,
            key=lambda household_id: int(
                household_id.split("_")[1]
            ),
        )
    )

    # Seasonal summary containers.
    dfs_by_season = {}
    annual_cooling_by_household = None


    # -------------------------------------------------------------------------
    #   Sec 3.7 -- 1.3) Initialize output writers
    # -------------------------------------------------------------------------
    writer_occ = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_occupancy.xlsx"), engine="openpyxl")

    dhw_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_DHW_consumption_kWh.xlsx"), engine="openpyxl")

    induction_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_induction_consumption_kWh.xlsx"), engine="openpyxl")

    # Greedy home-charging profile: charge as soon as the EV is at home and the minimum dwell time is satisfied.
    ev_home_charging_greedy_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_EV_home_greedy_charging_kWh.xlsx"), engine="openpyxl")

    # EV traction-energy demand generated while travelling. MODULE_04 uses this profile to reconstruct and optimize charging decisions.
    ev_demand_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_EV_electricity_demand_kWh.xlsx"), engine="openpyxl")
    ev_work_charging_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_EV_work_greedy_charging_kWh.xlsx"), engine="openpyxl")
    ev_soc_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_SoC_EV.xlsx"), engine="openpyxl")
    ev_km_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_EV_km_travelled.xlsx"), engine="openpyxl")

    cooling_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_Cooling_consumption_kWh.xlsx"), engine="openpyxl")

    heating_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_Heating_consumption_kWh.xlsx"), engine="openpyxl")

    additional_total_writer = pd.ExcelWriter(additional_weekly_aggregated_file, engine="openpyxl")

    household_annual_writer = pd.ExcelWriter(additional_annual_values_file, engine="openpyxl")
    household_timeseries_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_timeseries_kWh_per_household_TOT.xlsx"), engine="openpyxl")
    household_load_timeseries_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_ts_kWh_el_by_household_by_load.xlsx"), engine="openpyxl")

    dhw_overlap_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_DHW_overlap.xlsx"), engine="openpyxl")
    cooking_overlap_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f"{num_households}_households_{selected_region}_Cooking_overlap.xlsx"), engine="openpyxl")

    output_writers = (
        writer_occ,
        dhw_writer,
        induction_writer,
        ev_home_charging_greedy_writer,
        ev_demand_writer,
        ev_work_charging_writer,
        ev_soc_writer,
        ev_km_writer,
        cooling_writer,
        heating_writer,
        additional_total_writer,
        household_annual_writer,
        household_timeseries_writer,
        household_load_timeseries_writer,
        dhw_overlap_writer,
        cooking_overlap_writer,
    )

    load_order = [
        "DHW",
        "Cook",
        "Cool",
        "Heat",
        "EV_home",
    ]

    def household_load_sort_key(column):
        match = re.match(
            r"^(Household_\d+)_(.*)$",
            column,
        )

        if match is None:
            return (float("inf"), 999)

        household_id, load_name = match.groups()
        household_number = int(
            household_id.split("_")[1]
        )

        load_position = (
            load_order.index(load_name)
            if load_name in load_order
            else 999
        )

        return (
            household_number,
            load_position,
        )


    try:
        # =====================================================================
        #   Sec 3.7 -- 2) Calculate representative seasonal profiles
        # =====================================================================
        for season in selectable_periods:

            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.1) Load activity profiles and infer occupancy
            # -----------------------------------------------------------------
            activity_sheet_name = next(
                (
                    sheet_name
                    for sheet_name in sheets_dict
                    if canonical_season(sheet_name) == season
                ),
                None,
            )

            if activity_sheet_name is None:
                raise KeyError(
                    f"No activity-profile sheet found for season "
                    f"'{season}'. Available sheets: {list(sheets_dict)}"
                )

            activity_df = sheets_dict[
                activity_sheet_name
            ].copy()

            activity_df["Datetime"] = pd.to_datetime(
                activity_df["Datetime"]
            )
            activity_df.set_index(
                "Datetime",
                inplace=True,
            )
            activity_df = normalize_household_profile_columns(
                activity_df,
                default_season=season,
            )

            df_occ = activity_df.apply(
                infer_occupancy_hybrid_probabilistic
            )
            df_occ.to_excel(
                writer_occ,
                sheet_name=season,
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.2) Calculate DHW demand and resolve overlaps
            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.2.1) Generate RAW shower demand + audit of Personal-care events
            
            (raw_dhw_week_df, dhw_generation_audit_df,
            ) = calculate_dhw_consumption_by_season(
                activity_df,
                shower_presence_by_household,

                morning_shower_use_probability,
                daytime_shower_use_probability,
                evening_shower_use_probability,

                min_shower_consumption_kwh_th,
                max_shower_consumption_kwh_th,
                dhw_boiler_efficiency,

                personal_care_cluster_max_gap_ts=(
                    personal_care_cluster_max_gap_ts
                ),

                household_type_by_household=(
                    household_type_by_household
                ),
            )


            #   Sec 3.7 -- 2.2.2) Resolve simultaneous / closely-spaced showers + audit all shifts
            # Shift simultaneous or closely spaced showers when the required
            # energy is incompatible with the assumed ESWH capacity/recharge.
            (dhw_week_df, dhw_shift_audit_df,
            ) = resolve_dhw_overlap_by_household(
                raw_dhw_week_df,

                dhw_boiler_specs_by_household=(dhw_boiler_specs_by_household),

                simultaneous_probability=(simultaneous_shower_probability),

                step_minutes=step_minutes,

                alpha_util_boiler=(alpha_util_boiler),

                reference_shower_energy_kwh=(reference_shower_consumption_kwh_th),

                use_average_shower_for_shift=(use_average_shower_for_shift),

                gap_reset_cluster_ts=(gap_reset_cluster_ts),

                max_shift_steps=(max_shower_shift_steps),

                generation_audit_df=(dhw_generation_audit_df),
            )

            #   Sec 3.7 -- 2.2.3) Save final DHW demand
            dhw_week_df.to_excel(
                dhw_writer,
                sheet_name=season,
            )

            #   Sec 3.7 -- 2.2.4) SAVE DHW GENERATION / SHIFT AUDIT
            # =====================================================================

            # A) All reconstructed Personal-care events
            dhw_generation_audit_df.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_PC_events",
                index=False,
            )

            # B) One row for each RAW shower
            dhw_shift_audit_df.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_shower_audit",
                index=False,
            )

            # ---------------------------------------------------------------------
            # Prepare weekly shower counts per person
            # One row per generated shower, with its final timestamp after shifting.
            shower_events_df = dhw_shift_audit_df.copy()

            # Include all persons, including those with zero showers.
            all_persons = pd.Index(
                activity_df.columns,
                name="person",
            )

            showers_per_person = (
                shower_events_df
                .groupby("person")
                .size()
                .reindex(
                    all_persons,
                    fill_value=0,
                )
                .rename("n_showers_week")
                .reset_index()
            )


            # C) Aggregate summary
            n_pc_events = len(
                dhw_generation_audit_df
            )

            n_raw_showers = int(
                dhw_generation_audit_df[
                    "raw_shower_generated"
                ].sum()
            )

            n_shifted_showers = (
                int(
                    dhw_shift_audit_df[
                        "shifted"
                    ].sum()
                )
                if not dhw_shift_audit_df.empty
                else 0
            )


            summary_rows = [
                {
                    "metric": "personal_care_events",
                    "value": n_pc_events,
                },
                {
                    "metric": "raw_showers_generated",
                    "value": n_raw_showers,
                },
                {
                    "metric": "showers_shifted",
                    "value": n_shifted_showers,
                },
                {
                    "metric": "share_raw_showers_shifted_pct",
                    "value": (
                        100.0
                        * n_shifted_showers
                        / n_raw_showers
                        if n_raw_showers > 0
                        else 0.0
                    ),
                },
            ]


            if not dhw_shift_audit_df.empty:

                shifted_only = (
                    dhw_shift_audit_df[
                        dhw_shift_audit_df["shifted"]
                    ]
                )

                summary_rows.extend([
                    {
                        "metric": "mean_shift_steps_shifted",
                        "value": (
                            shifted_only[
                                "shift_steps_total"
                            ].mean()
                            if not shifted_only.empty
                            else 0.0
                        ),
                    },
                    {
                        "metric": "max_shift_steps",
                        "value": (
                            shifted_only[
                                "shift_steps_total"
                            ].max()
                            if not shifted_only.empty
                            else 0
                        ),
                    },
                    {
                        "metric": "shift_simultaneity_only",
                        "value": int(
                            (
                                dhw_shift_audit_df[
                                    "shift_reason"
                                ]
                                == "simultaneity"
                            ).sum()
                        ),
                    },
                    {
                        "metric": "shift_energy_budget_only",
                        "value": int(
                            (
                                dhw_shift_audit_df[
                                    "shift_reason"
                                ]
                                == "energy_budget"
                            ).sum()
                        ),
                    },
                    {
                        "metric": "shift_both_reasons",
                        "value": int(
                            (
                                dhw_shift_audit_df[
                                    "shift_reason"
                                ]
                                == "both"
                            ).sum()
                        ),
                    },
                ])


            summary_rows.extend([
                {
                    "metric": "n_persons",
                    "value": len(
                        showers_per_person
                    ),
                },
                {
                    "metric": "mean_showers_per_person_week",
                    "value": (
                        showers_per_person[
                            "n_showers_week"
                        ].mean()
                    ),
                },
                {
                    "metric": "median_showers_per_person_week",
                    "value": (
                        showers_per_person[
                            "n_showers_week"
                        ].median()
                    ),
                },
                {
                    "metric": "min_showers_per_person_week",
                    "value": (
                        showers_per_person[
                            "n_showers_week"
                        ].min()
                    ),
                },
                {
                    "metric": "max_showers_per_person_week",
                    "value": (
                        showers_per_person[
                            "n_showers_week"
                        ].max()
                    ),
                },
            ])





            dhw_audit_summary_df = pd.DataFrame(
                summary_rows
            )

            dhw_audit_summary_df.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_shower_summary",
                index=False,
            )

            # D) Summary by household type
            if not dhw_shift_audit_df.empty:

                dhw_audit_by_family_type_df = (
                    dhw_shift_audit_df
                    .groupby(
                        [
                            "household_type",
                            "family_size",
                        ],
                        dropna=False,
                    )
                    .agg(
                        raw_showers=(
                            "event_id",
                            "count",
                        ),

                        shifted_showers=(
                            "shifted",
                            "sum",
                        ),

                        mean_shift_steps=(
                            "shift_steps_total",
                            "mean",
                        ),

                        max_shift_steps=(
                            "shift_steps_total",
                            "max",
                        ),

                        mean_shower_energy_kwh_th=(
                            "shower_energy_kwh_th",
                            "mean",
                        ),

                        mean_pc_events_same_person_day=(
                            "n_personal_care_events_same_person_day",
                            "mean",
                        ),
                    )
                    .reset_index()
                )

                dhw_audit_by_family_type_df[
                    "share_showers_shifted_pct"
                ] = (
                    100.0
                    * dhw_audit_by_family_type_df[
                        "shifted_showers"
                    ]
                    / dhw_audit_by_family_type_df[
                        "raw_showers"
                    ]
                )

            else:

                dhw_audit_by_family_type_df = (
                    pd.DataFrame()
                )


            dhw_audit_by_family_type_df.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_by_family",
                index=False,
            )


            # =====================================================================
            # E) WEEKLY SHOWER FREQUENCY AND TIME-OF-DAY AUDIT
            # =====================================================================

            # Add household identifier.
            showers_per_person["household"] = (
                showers_per_person["person"]
                .astype(str)
                .str.extract(r"^(Household_\d+)")[0]
            )


            # Household type.
            showers_per_person["household_type"] = (
                showers_per_person["household"]
                .map(household_type_by_household)
                .fillna("Unknown")
            )


            # Real family size from activity-profile columns.
            person_household_series = (
                pd.Series(
                    activity_df.columns,
                    index=activity_df.columns,
                )
                .astype(str)
                .str.extract(r"^(Household_\d+)")[0]
            )

            family_size_by_household = (
                person_household_series
                .value_counts()
                .to_dict()
            )

            showers_per_person["family_size"] = (
                showers_per_person["household"]
                .map(family_size_by_household)
            )


            # Reorder columns.
            showers_per_person = showers_per_person[
                [
                    "person",
                    "household",
                    "household_type",
                    "family_size",
                    "n_showers_week",
                ]
            ]


            showers_per_person.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_showers_per_person",
                index=False,
            )

            # ---------------------------------------------------------------------
            # E2) Distribution of weekly showers per person
            # ---------------------------------------------------------------------

            weekly_shower_distribution = (
                showers_per_person[
                    "n_showers_week"
                ]
                .value_counts()
                .sort_index()
                .rename_axis(
                    "n_showers_week"
                )
                .rename(
                    "n_persons"
                )
                .reset_index()
            )

            weekly_shower_distribution[
                "share_persons_pct"
            ] = (
                100.0
                * weekly_shower_distribution[
                    "n_persons"
                ]
                / len(showers_per_person)
            )


            weekly_shower_distribution.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_showers_pp_dist",
                index=False,
            )

            # ---------------------------------------------------------------------
            # E3) Time-of-day distribution of FINAL showers
            # ---------------------------------------------------------------------

            if not shower_events_df.empty:

                shower_events_df[
                    "final_datetime"
                ] = pd.to_datetime(
                    shower_events_df[
                        "final_datetime"
                    ]
                )

                shower_events_df[
                    "final_hour"
                ] = (
                    shower_events_df[
                        "final_datetime"
                    ].dt.hour
                )


                # -------------------------------------------------------------
                # Probability bands used by the DHW generator:
                #
                # morning = 00:00-10:59
                # daytime = 11:00-17:59
                # evening = 18:00-23:59
                # -------------------------------------------------------------
                def classify_probability_period(hour):

                    if hour < 11:
                        return "morning"

                    elif hour < 18:
                        return "daytime"

                    else:
                        return "evening"


                shower_events_df[
                    "probability_period"
                ] = (
                    shower_events_df[
                        "final_hour"
                    ]
                    .apply(
                        classify_probability_period
                    )
                )


                # -------------------------------------------------------------
                # More detailed reporting bins for comparison with the
                # BVA-Doxa survey.
                #
                # IMPORTANT:
                # "after_work" and "before_bed" are TIME proxies only.
                # The survey categories are behavioural/self-reported categories.
                # -------------------------------------------------------------
                def classify_reporting_period(hour):

                    if hour < 11:
                        return "morning"

                    elif hour < 18:
                        return "daytime_other"

                    elif hour < 21:
                        return "after_work"

                    else:
                        return "before_bed"


                shower_events_df[
                    "reporting_period"
                ] = (
                    shower_events_df[
                        "final_hour"
                    ]
                    .apply(
                        classify_reporting_period
                    )
                )


                reporting_order = [
                    "morning",
                    "daytime_other",
                    "after_work",
                    "before_bed",
                ]


                shower_timing_summary = (
                    shower_events_df[
                        "reporting_period"
                    ]
                    .value_counts()
                    .reindex(
                        reporting_order,
                        fill_value=0,
                    )
                    .rename_axis(
                        "period"
                    )
                    .rename(
                        "n_showers"
                    )
                    .reset_index()
                )


                shower_timing_summary[
                    "share_showers_pct"
                ] = (
                    100.0
                    * shower_timing_summary[
                        "n_showers"
                    ]
                    / shower_timing_summary[
                        "n_showers"
                    ].sum()
                )


                period_ranges = {
                    "morning": "00:00-10:59",
                    "daytime_other": "11:00-17:59",
                    "after_work": "18:00-20:59",
                    "before_bed": "21:00-23:59",
                }

                shower_timing_summary[
                    "time_range"
                ] = (
                    shower_timing_summary[
                        "period"
                    ]
                    .map(period_ranges)
                )


            else:

                shower_timing_summary = pd.DataFrame(
                    columns=[
                        "period",
                        "n_showers",
                        "share_showers_pct",
                        "time_range",
                    ]
                )


            shower_timing_summary.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_shower_timing",
                index=False,
            )

            # ---------------------------------------------------------------------
            # E4) Hourly shower distribution
            # ---------------------------------------------------------------------

            if not shower_events_df.empty:

                shower_hourly_distribution = (
                    shower_events_df[
                        "final_hour"
                    ]
                    .value_counts()
                    .reindex(
                        range(24),
                        fill_value=0,
                    )
                    .rename_axis(
                        "hour"
                    )
                    .rename(
                        "n_showers"
                    )
                    .reset_index()
                )

                shower_hourly_distribution[
                    "share_showers_pct"
                ] = (
                    100.0
                    * shower_hourly_distribution[
                        "n_showers"
                    ]
                    / shower_hourly_distribution[
                        "n_showers"
                    ].sum()
                )

            else:

                shower_hourly_distribution = pd.DataFrame(
                    {
                        "hour": range(24),
                        "n_showers": 0,
                        "share_showers_pct": 0.0,
                    }
                )


            shower_hourly_distribution.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_shower_hourly",
                index=False,
            )

            # -----------------------------------


            # Diagnose simultaneous showers before post-processing.
            dhw_binary = (
                raw_dhw_week_df > 0
            ).astype(int)

            household_of_person = (
                dhw_binary.columns
                .to_series()
                .str.extract(r"^(Household_\d+)")[0]
            )

            dhw_count_by_household = (
                dhw_binary
                .T
                .groupby(household_of_person)
                .sum()
                .T
            )

            overlap_mask = (
                dhw_count_by_household >= 2
            )

            overlap_events = overlap_mask.stack()
            overlap_events = overlap_events[
                overlap_events
            ]

            df_events = overlap_events.index.to_frame(
                index=False
            )
            df_events.columns = [
                "Datetime",
                "Household",
            ]
            df_events["members_with_DHW"] = (
                dhw_count_by_household
                .stack()
                .reindex(overlap_events.index)
                .to_numpy()
            )

            count_by_household = (
                overlap_mask
                .sum(axis=0)
                .rename("n_timesteps_overlap")
            )

            df_events.to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_events_pre_process",
                index=False,
            )
            count_by_household.to_frame().to_excel(
                dhw_overlap_writer,
                sheet_name=f"{season}_counts_pre_process",
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.3) Calculate cooking demand and resolve overlaps
            # -----------------------------------------------------------------
            raw_induction_week_df = (
                calculate_cooking_consumption_by_season(
                    activity_df,
                    induction_presence_by_household,
                    cold_season_induction_use_probability,
                    induction_consumption_kwh,
                )
            )

            (
                df_cook_ts_events,
                cook_ts_count,
                df_cook_cluster_events,
                _,
                df_cook_summary,
            ) = diagnose_cooking_overlap_by_household(
                raw_induction_week_df,
                window_ts=cooking_window_steps,
            )

            df_cook_ts_events.to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_events_ts_pre",
                index=False,
            )
            cook_ts_count.to_frame().to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_counts_ts_pre",
            )
            df_cook_cluster_events.to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_cluster_pre",
                index=False,
            )
            df_cook_summary.to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_summary_pre",
            )

            induction_week_df = (
                resolve_cooking_overlap_by_household(
                    raw_induction_week_df,
                    window_ts=cooking_window_steps,
                    secondary_event_weight=(
                        simultaneous_cooking_probability
                    ),
                    main_event_criterion="max_energy",
                )
            )

            (
                df_cook_ts_events_post,
                cook_ts_count_post,
                df_cook_cluster_events_post,
                _,
                df_cook_summary_post,
            ) = diagnose_cooking_overlap_by_household(
                induction_week_df,
                window_ts=cooking_window_steps,
            )

            df_cook_ts_events_post.to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_events_ts_post",
                index=False,
            )
            cook_ts_count_post.to_frame().to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_counts_ts_post",
            )
            df_cook_cluster_events_post.to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_cluster_post",
                index=False,
            )
            df_cook_summary_post.to_excel(
                cooking_overlap_writer,
                sheet_name=f"{season}_summary_post",
            )

            induction_week_df.to_excel(
                induction_writer,
                sheet_name=season,
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.4) Calculate EV use and charging profiles
            # -----------------------------------------------------------------
            (
                df_charge_home_week,
                df_charge_work_week,
                df_soc,
                df_km,
                ev_demand_week_df,
            ) = calculate_ev_consumption_by_season(
                activity_df,
                df_occ,
                ev_presence_by_household,
                car_use_probability,
                km_per_step,
                consumption_by_household,
                battery_capacity_by_household,
                workplace_charging_probability,
                home_charging_power_kw,
                workplace_charging_power_kw,
                charging_efficiency,
                min_dwell_steps,
            )

            # Keep Member_1 columns belonging to EV-equipped households.
            columns_series = df_soc.columns.to_series()
            member_1_mask = columns_series.str.contains(
                r"_Member_1_"
            )
            household_for_column = columns_series.str.extract(
                r"^(Household_\d+)"
            )[0]
            has_ev_mask = (
                household_for_column
                .map(ev_presence_by_household)
                .fillna(False)
                .astype(bool)
            )

            ev_soc_columns = columns_series[
                member_1_mask & has_ev_mask
            ].tolist()

            # Retain only EV columns with non-zero travelled distance.
            km_sums = df_km[
                ev_soc_columns
            ].sum(axis=0)
            ev_km_columns = km_sums[
                km_sums > 0
            ].index.tolist()

            df_soc_filtered = df_soc.loc[
                :,
                ev_soc_columns,
            ]
            df_km_filtered = df_km.loc[
                :,
                ev_km_columns,
            ]

            df_soc_filtered.to_excel(
                ev_soc_writer,
                sheet_name=season,
            )
            df_km_filtered.to_excel(
                ev_km_writer,
                sheet_name=season,
            )
            df_charge_home_week.to_excel(
                ev_home_charging_greedy_writer,
                sheet_name=season,
            )
            df_charge_work_week.to_excel(
                ev_work_charging_writer,
                sheet_name=season,
            )
            ev_demand_week_df.to_excel(
                ev_demand_writer,
                sheet_name=season,
            )

            # Remove the profile suffix from person identifiers before adding
            # the representative-week distance to the annual accumulator.
            df_km_base = df_km_filtered.rename(
                columns=lambda column: re.match(
                    r"^(Household_\d+_Member_\d+)_",
                    column,
                ).group(1)
            )

            weekly_km = weekly_km.add(
                df_km_base.sum(axis=0),
                fill_value=0.0,
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.5) Calculate cooling demand
            # -----------------------------------------------------------------
            start_timestamp = (
                df_occ.index.min().normalize()
            )

            hourly_index = pd.date_range(
                start=start_timestamp,
                periods=168,
                freq="h",
            )

            cooling_column = find_compatible_column(
                cooling_week_df,
                season,
                COOLING_SEASON_COLUMN_ALIASES,
            )

            cooling_hourly_series = pd.Series(
                cooling_week_df[cooling_column].to_numpy(),
                index=hourly_index,
                name="cooling_per_person_kW",
            )

            # Spread each hourly value across six 10-minute timesteps.
            cooling_10min_series = (
                cooling_hourly_series
                .reindex(
                    index=df_occ.index,
                    method="ffill",
                    limit=6,
                )
                .div(6.0)
                .fillna(0.0)
            )

            cooling_base_df = pd.DataFrame(
                np.repeat(
                    cooling_10min_series
                    .to_numpy(dtype=float)[:, None],
                    df_occ.shape[1],
                    axis=1,
                ),
                index=df_occ.index,
                columns=df_occ.columns,
                dtype=float,
            )

            # Cooling is active only when the corresponding person is at home.
            df_cooling_week = cooling_base_df * df_occ
            df_cooling_week.to_excel(
                cooling_writer,
                sheet_name=season,
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.6) Calculate space-heating demand
            # -----------------------------------------------------------------
            df_heat_week = (
                calculate_heating_consumption_by_season(
                    thermal_profile_cop_excel_path,
                    annual_thermal_demand_th,
                    season,
                    season_periods,
                    df_occ,
                    min_absent_steps,
                )
            )

            df_heat_week.to_excel(
                heating_writer,
                sheet_name=season,
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.7) Aggregate load components by household
            # -----------------------------------------------------------------
            household_dhw_df = persons_df_to_household_df(
                dhw_week_df,
                household_index,
            )
            household_cooking_df = persons_df_to_household_df(
                induction_week_df,
                household_index,
            )
            household_cooling_df = persons_df_to_household_df(
                df_cooling_week,
                household_index,
            )

            # Two EV quantities are intentionally kept distinct:
            # - traction demand: battery energy consumed while driving;
            # - home charging: electricity drawn from the dwelling meter.
            household_ev_driving_demand_df = (
                persons_df_to_household_df(
                    ev_demand_week_df,
                    household_index,
                )
            )
            household_ev_home_charging_df = (
                persons_df_to_household_df(
                    df_charge_home_week,
                    household_index,
                )
            )

            # Heating is already calculated at household level.
            household_heating_df = (
                df_heat_week
                .reindex(
                    columns=household_index,
                    fill_value=0.0,
                )
                .astype(float)
            )

            # This workbook contains actual dwelling-meter loads. EV traction
            # demand remains available separately in ev_demand_writer.
            household_load_ts_df = pd.concat(
                [
                    household_dhw_df.rename(
                        columns=lambda column: f"{column}_DHW"
                    ),
                    household_cooking_df.rename(
                        columns=lambda column: f"{column}_Cook"
                    ),
                    household_cooling_df.rename(
                        columns=lambda column: f"{column}_Cool"
                    ),
                    household_heating_df.rename(
                        columns=lambda column: f"{column}_Heat"
                    ),
                    household_ev_home_charging_df.rename(
                        columns=lambda column: f"{column}_EV_home"
                    ),
                ],
                axis=1,
            )

            household_load_ts_df = household_load_ts_df.reindex(
                sorted(
                    household_load_ts_df.columns,
                    key=household_load_sort_key,
                ),
                axis=1,
            )

            household_load_ts_df.to_excel(
                household_load_timeseries_writer,
                sheet_name=season,
            )

            # Total additional electricity demand at the household meter.
            # EV home charging is included; EV traction demand is not a meter
            # load and is therefore excluded from this sum.
            household_load_components = [
                household_dhw_df,
                household_cooking_df,
                household_cooling_df,
                household_ev_home_charging_df,
                household_heating_df,
            ]

            component_arrays = [
                component_df.to_numpy(dtype=float)
                for component_df in household_load_components
            ]

            total_household_array = np.nansum(
                np.stack(
                    component_arrays,
                    axis=0,
                ),
                axis=0,
            )

            total_household_df = pd.DataFrame(
                total_household_array,
                index=household_dhw_df.index,
                columns=household_index,
            )

            total_household_df.to_excel(
                household_timeseries_writer,
                sheet_name=season,
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.8) Calculate seasonal and annualized indicators
            # -----------------------------------------------------------------
            dhw_person_week = dhw_week_df.sum(axis=0)
            cooking_person_week = induction_week_df.sum(axis=0)
            ev_home_charging_person_week = (
                df_charge_home_week.sum(axis=0)
            )
            km_person_week = df_km.sum(axis=0)

            household_dhw_week = (
                aggregate_person_to_household_sum(
                    dhw_person_week
                )
            )
            household_cooking_week = (
                aggregate_person_to_household_sum(
                    cooking_person_week
                )
            )
            household_ev_home_charging_week = (
                aggregate_person_to_household_sum(
                    ev_home_charging_person_week
                )
            )
            household_km_week = (
                aggregate_person_to_household_sum(
                    km_person_week
                )
            )

            # Each representative week represents 13 weeks of its season.
            household_dhw_season = (
                household_dhw_week * 13.0
            )
            household_cooking_season = (
                household_cooking_week * 13.0
            )
            household_ev_home_charging_season = (
                household_ev_home_charging_week * 13.0
            )
            household_km_season = (
                household_km_week * 13.0
            )

            household_heating_annual = (
                pd.Series(
                    annual_thermal_demand_th,
                    dtype=float,
                )
                .reindex(household_index)
                .fillna(0.0)
            )

            # Cooling is represented by the selected summer week, scaled to
            # the assumed 16-week cooling period.
            if season == "summer":
                cooling_person_week = (
                    df_cooling_week.sum(axis=0)
                )
                household_cooling_week = (
                    aggregate_person_to_household_sum(
                        cooling_person_week
                    )
                )
                annual_cooling_by_household = (
                    household_cooling_week
                    .mul(16.0)
                    .reindex(household_index)
                    .fillna(0.0)
                )

            if annual_cooling_by_household is not None:
                household_cooling_annual = (
                    annual_cooling_by_household
                    .reindex(household_index)
                    .fillna(0.0)
                )
            else:
                household_cooling_annual = pd.Series(
                    0.0,
                    index=household_index,
                )

            household_season_df = pd.DataFrame(
                index=household_index
            )
            household_season_df[
                "DHW_kWh_el_season"
            ] = household_dhw_season.reindex(
                household_index
            ).fillna(0.0)
            household_season_df[
                "Cooking_kWh_el_season"
            ] = household_cooking_season.reindex(
                household_index
            ).fillna(0.0)
            household_season_df[
                "EV_home_charging_kWh_el_season"
            ] = household_ev_home_charging_season.reindex(
                household_index
            ).fillna(0.0)
            household_season_df[
                "km_season"
            ] = household_km_season.reindex(
                household_index
            ).fillna(0.0)
            household_season_df[
                "Heating_kWh_th_annual"
            ] = household_heating_annual
            household_season_df[
                "Cooling_kWh_el_annual"
            ] = household_cooling_annual

            household_type_series = (
                pd.Series(household_type_by_household)
                .reindex(household_index)
                .fillna("Unknown")
            )

            household_season_df.insert(
                0,
                "Household type",
                household_type_series.to_numpy(),
            )

            season_columns = [
                "Household type",
                "DHW_kWh_el_season",
                "Cooking_kWh_el_season",
                "EV_home_charging_kWh_el_season",
                "km_season",
            ]

            household_season_reduced_df = (
                household_season_df
                .reindex(columns=season_columns)
                .fillna(0.0)
            )

            household_season_reduced_df.to_excel(
                household_annual_writer,
                sheet_name=season,
            )

            dfs_by_season[season] = (
                household_season_df.copy()
            )


            # -----------------------------------------------------------------
            #   Sec 3.7 -- 2.9) Save aggregated weekly additional loads
            # -----------------------------------------------------------------
            additional_loads_weekly_aggregated_ts_df = (
                pd.DataFrame(
                    {
                        "DHW": dhw_week_df.sum(axis=1),
                        "Cooking": induction_week_df.sum(axis=1),
                        "Cooling": df_cooling_week.sum(axis=1),
                        "EV": df_charge_home_week.sum(axis=1),
                        "Heating": df_heat_week.sum(axis=1),
                    }
                )
            )

            additional_loads_weekly_aggregated_ts_df.to_excel(
                additional_total_writer,
                sheet_name=season,
            )


        # =====================================================================
        #   Sec 3.7 -- 3) Build and save the annual household summary
        # =====================================================================
        annual_df = pd.DataFrame(
            index=household_index
        )

        seasonal_columns_map = {
            "DHW_kWh_el_annual": "DHW_kWh_el_season",
            "Cooking_kWh_el_annual": (
                "Cooking_kWh_el_season"
            ),
            "EV_home_charging_kWh_el_annual": (
                "EV_home_charging_kWh_el_season"
            ),
            "annual_km": "km_season",
        }

        for (
            annual_column,
            seasonal_column,
        ) in seasonal_columns_map.items():
            annual_values = pd.Series(
                0.0,
                index=household_index,
            )

            for season_df in dfs_by_season.values():
                if seasonal_column not in season_df.columns:
                    continue

                annual_values = annual_values.add(
                    season_df[seasonal_column]
                    .reindex(household_index)
                    .fillna(0.0)
                    .astype(float),
                    fill_value=0.0,
                )

            annual_df[annual_column] = annual_values

        annual_df["Heating_kWh_th_annual"] = (
            pd.Series(
                annual_thermal_demand_th,
                dtype=float,
            )
            .reindex(household_index)
            .fillna(0.0)
        )

        if annual_cooling_by_household is not None:
            annual_cooling_series = (
                annual_cooling_by_household
                .reindex(household_index)
                .fillna(0.0)
            )
        else:
            annual_cooling_series = pd.Series(
                0.0,
                index=household_index,
            )

        annual_df["Cooling_kWh_el_annual"] = (
            annual_cooling_series
        )

        annual_household_type_series = (
            pd.Series(household_type_by_household)
            .reindex(household_index)
            .fillna("Unknown")
        )

        annual_df.insert(
            0,
            "Household type",
            annual_household_type_series.to_numpy(),
        )

        annual_df.to_excel(
            household_annual_writer,
            sheet_name="Annual",
        )


        # =====================================================================
        #   Sec 3.7 -- 4) Save annual distance travelled by individual EV users
        # =====================================================================
        # The four representative weeks jointly cover 52 weeks; therefore each
        # accumulated representative-week contribution is scaled by 13.
        annual_km = weekly_km.mul(13.0)
        annual_km.name = "annual_km"
        annual_km = annual_km[
            annual_km > 0
        ]

        annual_km.to_frame().to_excel(
            os.path.join(
                additional_consumption_output_xls_folder,
                (
                    f"{num_households}_households_{selected_region}_"
                    "annual_km_EV.xlsx"
                ),
            ),
            sheet_name="annual",
        )


    finally:
        # =====================================================================
        #   Sec 3.7 -- 5) Close all output writers
        # =====================================================================
        for writer in output_writers:
            writer.close()


## 3.8 Additional-load plots and annual statistics

This section reads the outputs saved by Section 3.7 and independently generates
weekly stacked profiles, daily activity-consumption plots, annual percentile
plots and optional statistical tables.

BESS state-of-charge and optimization-result plots are generated downstream,
after running the Module IV (operation optimization of the energy system).

In [ ]:
if RUN_ADDITIONAL_LOAD_PLOTS:

    selected_plot_flags = (
        PLOT_WEEKLY_ADDITIONAL_STACKED,
        PLOT_STACKED_CONV_PLUS_ADDITIONAL,
        RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT,
        PLOT_ANNUAL_PERCENTILES_ALL_HOUSEHOLDS,
        PLOT_ANNUAL_PERCENTILES_BY_HOUSEHOLD_TYPE,
    )

    selected_postprocessing_outputs = (
        selected_plot_flags
        + (SAVE_ANNUAL_STATISTICS_TABLE,)
    )

    if not any(selected_postprocessing_outputs):
        print(
            "RUN_ADDITIONAL_LOAD_PLOTS is enabled, "
            "but no plot or statistical output is selected."
        )


    # =====================================================================
    #   Sec 3.8 -- 1) Weekly time-series plots
    # =====================================================================

    weekly_plot_requested = any(
        (
            PLOT_WEEKLY_ADDITIONAL_STACKED,
            PLOT_STACKED_CONV_PLUS_ADDITIONAL,
            RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT,
        )
    )

    if weekly_plot_requested:

        if not os.path.isfile(
            additional_weekly_aggregated_file
        ):
            raise FileNotFoundError(
                "Weekly aggregated additional-load file not found:\n"
                f"{additional_weekly_aggregated_file}\n"
                "Run Section 3.7 first."
            )

        additional_load_xls = pd.ExcelFile(
            additional_weekly_aggregated_file
        )

        additional_sheet_by_season = {
            canonical_season(sheet_name): sheet_name
            for sheet_name
            in additional_load_xls.sheet_names
        }

        base_load_xls = None

        base_load_required = (
            PLOT_STACKED_CONV_PLUS_ADDITIONAL
            or RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT
        )

        if base_load_required:
            base_load_path = (
                _resolve_base_load_excel_path()
            )

            if base_load_path is not None:
                base_load_xls = pd.ExcelFile(
                    base_load_path
                )

        activity_sheets = None

        if RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT:
            if not os.path.isfile(
                household_profiles_file
            ):
                raise FileNotFoundError(
                    "Household activity-profile file not found:\n"
                    f"{household_profiles_file}"
                )

            activity_sheets = pd.read_excel(
                household_profiles_file,
                sheet_name=None,
            )

        for season in selectable_periods:

            additional_sheet_name = (
                additional_sheet_by_season.get(
                    season
                )
            )

            if additional_sheet_name is None:
                print(
                    f"[WARN][PLOTS] No additional-load "
                    f"sheet found for '{season}'."
                )
                continue

            additional_week_df = pd.read_excel(
                additional_load_xls,
                sheet_name=additional_sheet_name,
                index_col=0,
                parse_dates=True,
            )

            additional_week_df.index = pd.to_datetime(
                additional_week_df.index
            )

            additional_week_df = (
                additional_week_df
                .apply(
                    pd.to_numeric,
                    errors="coerce",
                )
                .fillna(0.0)
            )


            # -------------------------------------------------------------
            #   Sec 3.8 -- 1.1) Weekly stacked additional loads only
            # -------------------------------------------------------------

            if PLOT_WEEKLY_ADDITIONAL_STACKED:

                additional_column_order = [
                    column
                    for column
                    in STACKED_CONSUMPTION_ORDER
                    if column != BASE_LOAD_LABEL
                ]

                title = (
                    f"Additional electricity demand — "
                    f"{season_english_dict[season]}\n"
                    f"{num_households} households - "
                    f"Zone: {selected_region}"
                )

                for layout in (
                    STACKED_CONSUMPTION_PLOT_LAYOUTS
                ):
                    if layout not in (
                        STACKED_CONSUMPTION_LAYOUT_PRESETS
                    ):
                        raise ValueError(
                            f"Unknown stacked-plot layout "
                            f"'{layout}'."
                        )

                    layout_options = (
                        STACKED_CONSUMPTION_LAYOUT_PRESETS[
                            layout
                        ]
                    )

                    plot_stacked_consumption_for_paper(
                        df_plot=additional_week_df,
                        title=title,
                        file_name=(
                            f"{num_households}_households_"
                            f"{selected_region}_"
                            f"weekly_additional_loads_"
                            f"{season}_{layout}.png"
                        ),
                        output_folder=(
                            additional_consumption_output_pictures_folder
                        ),
                        column_order=(
                            additional_column_order
                        ),
                        color_map=(
                            STACKED_CONSUMPTION_COLOR_MAP
                        ),
                        show_title=(
                            SHOW_STACKED_CONSUMPTION_TITLE
                        ),
                        y_label="Energy demand [kWh]",
                        save_pdf=(
                            SAVE_STACKED_CONSUMPTION_PDF
                        ),
                        **layout_options,
                    )


            # -------------------------------------------------------------
            #   Sec 3.8 -- 1.2) Build conventional + additional DataFrame
            # -------------------------------------------------------------

            conventional_plus_additional_df = None

            if (
                base_load_required
                and base_load_xls is not None
            ):
                base_total = (
                    _read_base_load_total_for_season(
                        base_load_xls=base_load_xls,
                        season=season,
                        target_index=(
                            additional_week_df.index
                        ),
                        sheet_kind=(
                            BASE_LOAD_SHEET_KIND
                        ),
                    )
                )

                if base_total is not None:
                    conventional_plus_additional_df = (
                        pd.concat(
                            [
                                base_total.rename(
                                    BASE_LOAD_LABEL
                                ),
                                additional_week_df,
                            ],
                            axis=1,
                        )
                        .fillna(0.0)
                    )


            # -------------------------------------------------------------
            #   Sec 3.8 -- 1.3) Weekly conventional + additional stacked plot
            # -------------------------------------------------------------

            if (
                PLOT_STACKED_CONV_PLUS_ADDITIONAL
                and conventional_plus_additional_df
                is not None
            ):
                title = (
                    f"Conventional and additional "
                    f"electricity demand — "
                    f"{season_english_dict[season]}\n"
                    f"{num_households} households - "
                    f"Zone: {selected_region}"
                )

                for layout in (
                    STACKED_CONSUMPTION_PLOT_LAYOUTS
                ):
                    if layout not in (
                        STACKED_CONSUMPTION_LAYOUT_PRESETS
                    ):
                        raise ValueError(
                            f"Unknown stacked-plot layout "
                            f"'{layout}'."
                        )

                    layout_options = (
                        STACKED_CONSUMPTION_LAYOUT_PRESETS[
                            layout
                        ]
                    )

                    plot_stacked_consumption_for_paper(
                        df_plot=(
                            conventional_plus_additional_df
                        ),
                        title=title,
                        file_name=(
                            f"{num_households}_households_"
                            f"{selected_region}_"
                            f"weekly_conventional_plus_"
                            f"additional_{season}_"
                            f"{layout}.png"
                        ),
                        output_folder=(
                            additional_consumption_output_pictures_folder
                        ),
                        column_order=(
                            STACKED_CONSUMPTION_ORDER
                        ),
                        color_map=(
                            STACKED_CONSUMPTION_COLOR_MAP
                        ),
                        show_title=(
                            SHOW_STACKED_CONSUMPTION_TITLE
                        ),
                        y_label="Energy demand [kWh]",
                        save_pdf=(
                            SAVE_STACKED_CONSUMPTION_PDF
                        ),
                        **layout_options,
                    )


            # -------------------------------------------------------------
            #   Sec 3.8 -- 1.4) Daily activity + consumption plot
            # -------------------------------------------------------------

            if (
                RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT
                and conventional_plus_additional_df
                is not None
                and _season_selected_for_daily_plot(
                    season,
                    DAILY_ACTIVITY_CONSUMPTION_PLOT_SEASONS,
                )
            ):
                activity_sheet_name = next(
                    (
                        sheet_name
                        for sheet_name in activity_sheets
                        if canonical_season(
                            sheet_name
                        ) == season
                    ),
                    None,
                )

                if activity_sheet_name is None:
                    print(
                        f"[WARN][DAILY PLOT] "
                        f"No activity sheet found for "
                        f"'{season}'."
                    )
                    continue

                activity_week_df = activity_sheets[
                    activity_sheet_name
                ].copy()

                activity_week_df = (
                    normalize_household_profile_columns(
                        activity_week_df,
                        default_season=season,
                    )
                )

                selected_date = (
                    DAILY_ACTIVITY_CONSUMPTION_DATE_BY_SEASON
                    .get(
                        season,
                        None,
                    )
                )

                title = (
                    f"Daily activities and electricity "
                    f"demand — "
                    f"{season_english_dict[season]}\n"
                    f"{num_households} households - "
                    f"Zone: {selected_region}"
                )

                file_name = (
                    f"{num_households}_households_"
                    f"{selected_region}_"
                    f"daily_activity_and_consumption_"
                    f"{season}_day"
                    f"{DAILY_ACTIVITY_CONSUMPTION_DAY_INDEX + 1}_"
                    f"two_column.png"
                )

                plot_daily_activity_and_consumption(
                    activity_week_df=activity_week_df,
                    consumption_week_df=(
                        conventional_plus_additional_df
                    ),
                    title=title,
                    file_name=file_name,
                    output_folder=(
                        additional_consumption_output_pictures_folder
                    ),
                    selected_date=selected_date,
                    day_index=(
                        DAILY_ACTIVITY_CONSUMPTION_DAY_INDEX
                    ),
                    activity_categories=(
                        aggregated_activities
                    ),
                    activity_code_to_label=(
                        activity_code_to_label
                    ),
                    activity_english_dict=(
                        activity_english_dict
                    ),
                    activity_color_map=(
                        activity_color_map
                    ),
                    consumption_column_order=(
                        STACKED_CONSUMPTION_ORDER
                    ),
                    consumption_color_map=(
                        STACKED_CONSUMPTION_COLOR_MAP
                    ),
                    expected_steps_per_day=144,
                    show_title=(
                        SHOW_DAILY_ACTIVITY_CONSUMPTION_TITLE
                    ),
                    fig_size=(7.20, 5.80),
                    font_size=9.0,
                    label_font_size=10.0,
                    tick_font_size=8.5,
                    activity_legend_font_size=6.2,
                    consumption_legend_font_size=8.0,
                    title_font_size=11.0,
                    save_pdf=(
                        SAVE_DAILY_ACTIVITY_CONSUMPTION_PDF
                    ),
                )

        additional_load_xls.close()

        if base_load_xls is not None:
            base_load_xls.close()

    # =====================================================================
    #   Sec 3.8 --  2) Annual percentile plots
    # =====================================================================

    annual_postprocessing_requested = any(
        (
            PLOT_ANNUAL_PERCENTILES_ALL_HOUSEHOLDS,
            PLOT_ANNUAL_PERCENTILES_BY_HOUSEHOLD_TYPE,
            SAVE_ANNUAL_STATISTICS_TABLE,
        )
    )

    if annual_postprocessing_requested:

        if not os.path.isfile(
            additional_annual_values_file
        ):
            raise FileNotFoundError(
                "Annual additional-load summary not found:\n"
                f"{additional_annual_values_file}\n"
                "Run Section 3.7 first."
            )

        annual_df = pd.read_excel(
            additional_annual_values_file,
            sheet_name="Annual",
            index_col=0,
        )

        if SAVE_ANNUAL_STATISTICS_TABLE:
            (
                overall_statistics_df,
                household_type_statistics_df,
            ) = build_annual_statistics_tables(
                annual_df=annual_df,
                metric_specs=ANNUAL_METRIC_SPECS,
                percentile_levels=PERCENTILE_LEVELS,
                household_type_column="Household type",
            )

            with pd.ExcelWriter(
                additional_annual_statistics_file,
                engine="openpyxl",
            ) as writer:

                overall_statistics_df.to_excel(
                    writer,
                    sheet_name="All households",
                    index=False,
                )

                household_type_statistics_df.to_excel(
                    writer,
                    sheet_name="By household type",
                    index=False,
                )

            print(
                "Annual additional-load statistics saved to:\n"
                f"{additional_annual_statistics_file}"
            )




        for (
            value_column,
            metric_spec,
        ) in ANNUAL_METRIC_SPECS.items():

            if value_column not in annual_df.columns:
                print(
                    f"[WARN][PERCENTILES] Column "
                    f"'{value_column}' not found."
                )
                continue

            metric_label = metric_spec["label"]
            metric_unit = metric_spec["unit"]
            file_tag = metric_spec["file_tag"]

            if (
                PLOT_ANNUAL_PERCENTILES_ALL_HOUSEHOLDS
            ):
                plot_annual_percentiles_all_households(
                    series=annual_df[value_column],
                    metric_label=metric_label,
                    metric_unit=metric_unit,
                    file_name=(
                        f"{num_households}_households_"
                        f"{selected_region}_"
                        f"annual_percentiles_"
                        f"{file_tag}_all_households.png"
                    ),
                    output_folder=(
                        additional_consumption_output_pictures_folder
                    ),
                    percentile_levels=(
                        PERCENTILE_LEVELS
                    ),
                    percentile_bands=(
                        PERCENTILE_BANDS
                    ),
                    show_title=True,
                    save_pdf=(
                        SAVE_ANNUAL_PERCENTILE_PDF
                    ),
                )

            if (
                PLOT_ANNUAL_PERCENTILES_BY_HOUSEHOLD_TYPE
            ):
                plot_annual_percentiles_by_household_type(
                    annual_df=annual_df,
                    value_column=value_column,
                    metric_label=metric_label,
                    metric_unit=metric_unit,
                    file_name=(
                        f"{num_households}_households_"
                        f"{selected_region}_"
                        f"annual_percentiles_"
                        f"{file_tag}_by_household_type.png"
                    ),
                    output_folder=(
                        additional_consumption_output_pictures_folder
                    ),
                    percentile_levels=(
                        PERCENTILE_LEVELS
                    ),
                    percentile_bands=(
                        PERCENTILE_BANDS
                    ),
                    household_type_column=(
                        "Household type"
                    ),
                    show_title=True,
                    save_pdf=(
                        SAVE_ANNUAL_PERCENTILE_PDF
                    ),
                )

## 3.9 Diagnostic on existing EV mobility-profile

In [ ]:
# =============================================================================
# Sec 3.9 -- Existing EV mobility-profile feasibility audit
# =============================================================================
#
# READ-ONLY diagnostic:
# - reads already generated MODULE_03 EV demand profiles;
# - reads the corresponding EV metadata;
# - identifies continuous EV mobility-demand blocks;
# - evaluates their energy requirement relative to the assigned battery;
# - DOES NOT rebuild or overwrite existing demand profiles.
# =============================================================================

if RUN_EV_TRIP_EXTREME_AUDIT:

    # -------------------------------------------------------------------------
    # Sec 3.9 -- 1) Audit settings
    # -------------------------------------------------------------------------

    # Keep this consistent with MODULE_04.
    EV_DISCHARGE_EFFICIENCY_AUDIT = 0.90

    # Thresholds reported in the diagnostic.
    EXTREME_TRIP_SOC_THRESHOLDS_PCT = [
        50.0,
        70.0,
        80.0,
        90.0,
        95.0,
        100.0,
    ]

    eps = 1e-12


    # -------------------------------------------------------------------------
    # Sec 3.9 -- 2) Existing files
    # -------------------------------------------------------------------------

    ev_demand_file = os.path.join(
        additional_consumption_output_xls_folder,
        (
            f"{num_households}_households_"
            f"{selected_region}_"
            f"EV_electricity_demand_kWh.xlsx"
        ),
    )

    ev_metadata_file = os.path.join(
        additional_consumption_output_xls_folder,
        (
            f"{num_households}_households_"
            f"{selected_region}_"
            f"ev_metadata_by_person.xlsx"
        ),
    )


    for required_file in (
        ev_demand_file,
        ev_metadata_file,
    ):
        if not os.path.isfile(required_file):
            raise FileNotFoundError(
                "Required existing MODULE_03 EV file not found:\n"
                f"{required_file}"
            )


    print(
        "\n[EV TRIP AUDIT] Existing profiles only."
        "\nNo MODULE_03 demand profile will be rebuilt or overwritten."
    )

    print(
        f"\nEV demand file:\n{ev_demand_file}"
    )

    print(
        f"\nEV metadata file:\n{ev_metadata_file}"
    )


    # -------------------------------------------------------------------------
    # Sec 3.9 -- 3) Load EV metadata
    # -------------------------------------------------------------------------

    ev_metadata = pd.read_excel(
        ev_metadata_file,
        index_col=0,
    )

    required_metadata_columns = {
        "household",
        "has_ev",
        "bev_capacity_kWh",
    }

    missing_metadata_columns = (
        required_metadata_columns
        - set(ev_metadata.columns)
    )

    if missing_metadata_columns:
        raise KeyError(
            "Missing columns in EV metadata: "
            f"{sorted(missing_metadata_columns)}"
        )


    # One battery capacity per EV-equipped household.
    household_capacity = (
        ev_metadata
        .loc[
            ev_metadata["has_ev"].astype(bool)
            & ev_metadata["bev_capacity_kWh"].notna()
        ]
        .groupby("household")["bev_capacity_kWh"]
        .first()
        .astype(float)
    )


    # -------------------------------------------------------------------------
    # Sec 3.9 -- 4) Load already generated seasonal EV demand profiles
    # -------------------------------------------------------------------------

    ev_demand_by_season = pd.read_excel(
        ev_demand_file,
        sheet_name=None,
        index_col=0,
    )


    trip_rows = []


    # -------------------------------------------------------------------------
    # Sec 3.9 -- 5) Identify continuous mobility-demand blocks
    # -------------------------------------------------------------------------

    for season, ev_df_raw in ev_demand_by_season.items():

        ev_df = (
            ev_df_raw
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .fillna(0.0)
        )

        # Recover datetime index when possible.
        try:
            ev_df.index = pd.to_datetime(
                ev_df.index
            )
        except Exception:
            pass


        for person in ev_df.columns:

            demand = (
                pd.to_numeric(
                    ev_df[person],
                    errors="coerce",
                )
                .fillna(0.0)
            )


            # Skip columns with no EV mobility demand.
            if float(demand.sum()) <= eps:
                continue


            # Extract household identifier from the person column.
            household_match = re.match(
                r"^(Household_\d+)",
                str(person),
            )

            if household_match is None:
                print(
                    "[WARN][EV TRIP AUDIT] "
                    f"Cannot identify household from column: {person}"
                )
                continue

            household = household_match.group(1)


            if household not in household_capacity.index:
                print(
                    "[WARN][EV TRIP AUDIT] "
                    f"No battery capacity for {household}"
                )
                continue


            capacity_kwh = float(
                household_capacity.loc[
                    household
                ]
            )


            # True when the existing MODULE_03 profile contains
            # positive EV traction demand.
            moving = demand > eps


            # Identify contiguous positive-demand blocks.
            block_id = (
                moving
                .ne(
                    moving.shift(
                        fill_value=False
                    )
                )
                .cumsum()
            )

            active_block_ids = (
                block_id[moving]
                .unique()
            )


            for trip_number, current_block in enumerate(
                active_block_ids,
                start=1,
            ):

                trip_mask = (
                    moving
                    & (
                        block_id
                        == current_block
                    )
                )

                trip_positions = np.flatnonzero(
                    trip_mask.to_numpy()
                )

                if len(trip_positions) == 0:
                    continue


                i_start = int(
                    trip_positions[0]
                )

                i_end = int(
                    trip_positions[-1]
                )


                trip_start = demand.index[
                    i_start
                ]

                trip_end = demand.index[
                    i_end
                ]


                # MODULE_03 profile:
                # traction energy requested by the trip.
                trip_traction_kwh = float(
                    demand.iloc[
                        i_start:
                        i_end + 1
                    ].sum()
                )


                # Direct ratio of MODULE_03 traction energy
                # to battery capacity.
                raw_trip_capacity_fraction = (
                    trip_traction_kwh
                    / capacity_kwh
                )


                # MODULE_04 currently applies discharge efficiency
                # when translating traction demand into battery-side
                # energy withdrawal.
                trip_battery_required_kwh = (
                    trip_traction_kwh
                    / float(
                        EV_DISCHARGE_EFFICIENCY_AUDIT
                    )
                )


                required_soc_pct = (
                    100.0
                    * trip_battery_required_kwh
                    / capacity_kwh
                )


                # With the current MODULE_04 GenericStorage
                # min_storage_level_BEV = 5%, the maximum usable
                # fraction from a full battery is 95%.
                required_departure_soc_with_5pct_floor = (
                    required_soc_pct
                    + 5.0
                )


                trip_rows.append(
                    {
                        "season":
                            season,

                        "household":
                            household,

                        "person":
                            person,

                        "trip_number":
                            trip_number,

                        "trip_start":
                            trip_start,

                        "trip_end":
                            trip_end,

                        "n_timesteps":
                            len(
                                trip_positions
                            ),

                        "battery_capacity_kWh":
                            capacity_kwh,

                        "trip_traction_kWh":
                            trip_traction_kwh,

                        "trip_traction_pct_capacity":
                            (
                                100.0
                                * raw_trip_capacity_fraction
                            ),

                        "trip_battery_required_kWh":
                            trip_battery_required_kwh,

                        "required_SOC_pct":
                            required_soc_pct,

                        "required_departure_SOC_with_5pct_floor":
                            required_departure_soc_with_5pct_floor,

                        "feasible_with_full_battery_SOCmin5":
                            bool(
                                required_soc_pct
                                <= 95.0 + 1e-9
                            ),

                        "requires_gt_80pct":
                            bool(
                                required_soc_pct
                                > 80.0
                            ),

                        "requires_gt_90pct":
                            bool(
                                required_soc_pct
                                > 90.0
                            ),

                        "requires_gt_95pct":
                            bool(
                                required_soc_pct
                                > 95.0
                            ),

                        "requires_gt_100pct":
                            bool(
                                required_soc_pct
                                > 100.0
                            ),
                    }
                )


    df_ev_trip_audit = pd.DataFrame(
        trip_rows
    )


    if df_ev_trip_audit.empty:

        print(
            "\n[EV TRIP AUDIT] "
            "No positive EV mobility-demand blocks found."
        )

    else:

        # Sort from the most demanding trip.
        df_ev_trip_audit = (
            df_ev_trip_audit
            .sort_values(
                "required_SOC_pct",
                ascending=False,
            )
            .reset_index(
                drop=True
            )
        )


        # ---------------------------------------------------------------------
        # Sec 3.9 -- 6) Summary
        # ---------------------------------------------------------------------

        n_trips = len(
            df_ev_trip_audit
        )

        summary_rows = []

        for threshold in (
            EXTREME_TRIP_SOC_THRESHOLDS_PCT
        ):

            mask = (
                df_ev_trip_audit[
                    "required_SOC_pct"
                ]
                >= threshold
            )

            summary_rows.append(
                {
                    "required_SOC_threshold_pct":
                        threshold,

                    "n_trips_at_or_above_threshold":
                        int(
                            mask.sum()
                        ),

                    "share_of_all_trips_pct":
                        (
                            100.0
                            * float(
                                mask.sum()
                            )
                            / float(
                                n_trips
                            )
                        ),
                }
            )


        df_ev_trip_audit_summary = (
            pd.DataFrame(
                summary_rows
            )
        )


        print(
            "\n=== EV TRIP ENERGY / SOC AUDIT SUMMARY ==="
        )

        print(
            f"Total continuous EV mobility blocks: {n_trips}"
        )

        print(
            "Maximum required SOC: "
            f"{df_ev_trip_audit['required_SOC_pct'].max():.2f}%"
        )

        print(
            "Median required SOC: "
            f"{df_ev_trip_audit['required_SOC_pct'].median():.2f}%"
        )

        print(
            "95th percentile required SOC: "
            f"{df_ev_trip_audit['required_SOC_pct'].quantile(0.95):.2f}%"
        )

        print(
            "99th percentile required SOC: "
            f"{df_ev_trip_audit['required_SOC_pct'].quantile(0.99):.2f}%"
        )


        display(
            df_ev_trip_audit_summary
        )


        # ---------------------------------------------------------------------
        # Sec 3.9 -- 7) Show only extreme trips
        # ---------------------------------------------------------------------

        df_ev_trip_extreme = (
            df_ev_trip_audit.loc[
                df_ev_trip_audit[
                    "required_SOC_pct"
                ]
                >= 80.0
            ]
            .copy()
        )


        print(
            "\n=== EV TRIPS REQUIRING >= 80% SOC ==="
        )

        display(
            df_ev_trip_extreme
        )


        print(
            "\n=== EV TRIPS INCOMPATIBLE WITH SOC_min = 5% ==="
        )

        display(
            df_ev_trip_audit.loc[
                ~df_ev_trip_audit[
                    "feasible_with_full_battery_SOCmin5"
                ]
            ]
        )

## Conclusion

In [ ]:
#%% ################# CONCLUSION #######################################

# Get the end time.
end_time = datetime.datetime.now()
print(f"End time : {end_time}")

# Calculate execution time.
execution_time = end_time - start_time
print(f"Execution time : {execution_time}")